# Costruzione del panel — versione leggera

Costruisce il panel azienda-anno leggendo i CSV grezzi di PitchBook: una riga
per ogni anno di vita di ogni azienda, dall'anno di fondazione all'ultimo anno
in cui esiste un dato su di lei.

Produce **52 colonne**: 50 di `data/raw/example_panel.csv` — con `TotalRaised`
al posto di `TotalRaised_Est` e senza le tre `*_All` — piu' `UndisclosedAmountShare`
(blocco 4.8) e `N_Similar` (blocco 7.4).

Le fasi, in ordine: **1** aziende e scheletro · **2a** la tabella persona-azienda
· **2b** le colonne di team · **4** deal e investitori · **5** finalizzazione ·
**6** raggruppamento degli stadi e troncamento · **7** competitor.

Gli output di stadio vanno in `data/interim_light/`; il panel finale e'
`panel.parquet` e `panel.csv.gz`.

### Una scelta di lettura: i token che valgono «mancante»

Ogni CSV viene letto come testo e poi passato in `as_na` con **`R_NA_NAN`**
(`""`, `"NA"`, `"N/A"`, `"NULL"`, `"NaN"`): senza questo passaggio un segnaposto
testuale diventerebbe una categoria a se' in una colonna categoriale.
*Verificato: nelle nove tabelle lette il token `"NaN"` non compare in nessuna
cella, quindi la scelta non sposta un solo valore.*

**`R_NA_INF`** e' un insieme diverso e si applica **dopo** le aggregazioni: serve
a convertire i `-Inf` prodotti da `max()` e i `NaN` prodotti da `mean()` sui
gruppi vuoti, che sono artefatti del calcolo e non valori letti dai file.

In [1]:
# ── Import ────────────────────────────────────────────────────────────────
import dataclasses                    # serve solo a clonare PanelConfig cambiandone un campo
import gc                             # garbage collection manuale fra uno stadio e l'altro
import sys
from pathlib import Path

import polars as pl
import yaml

ROOT = Path.cwd()                     # la cartella da cui gira il notebook = radice del repo
if str(ROOT) not in sys.path:         # senza questo, "from src.panel..." non trova i moduli
    sys.path.insert(0, str(ROOT))

from src.panel.config import PanelConfig
from src.panel.expansions import active_pairs, expand_team
from src.panel.io import COMPANY_DATE_COLUMNS, read_raw, to_num
from src.panel.rutils import (
    # I due insiemi di token che valgono "mancante". Fanno due lavori diversi:
    R_NA_NAN,              # ("", "NA", "N/A", "NULL", "NaN") -> in LETTURA, su ogni CSV
    R_NA_INF,              # i precedenti + "Inf", "-Inf"     -> DOPO le aggregazioni
    as_na,                 # applica quei token a tutte le colonne di un frame
    cumany,                # flag cumulativo: una volta acceso, resta acceso
    next_different,        # primo valore futuro diverso dal corrente, e a che distanza
    parse_date_r,          # parser di date: il formato dipende dalla lunghezza
    r_case_when,           # catena di grepl a corto circuito: il primo che matcha vince
    r_cum_sum,             # somma cumulata che propaga i null fino in fondo al gruppo
    r_if_else,             # if/else con condizione nulla -> risultato nullo
    r_seq,                 # seq(): inclusivo, e conta ALL'INDIETRO se la fine precede l'inizio
    tail_na_omit,          # tail(na.omit(x), 1): l'ultimo valore non nullo
    weighted_cumulative,   # media ponderata cumulata
)

# interim_dir separata: il confronto finale ha bisogno che i parquet del panel
# completo, in data/interim/, restino intatti.
cfg = dataclasses.replace(PanelConfig(), interim_dir=Path("data/interim_light"))

pl.Config.set_tbl_cols(12)            # quante colonne mostrare quando si stampa un frame
pl.Config.set_fmt_str_lengths(40)

# ── La soglia di inclusione, letta dalla configurazione ───────────────────
# ANNO_MIN_FONDAZIONE e' l'anno di fondazione piu' antico ammesso nel panel,
# ed e' INCLUSIVO: con 2000, un'azienda fondata nel 2000 entra. Vive in
# config/config.yaml perche' e' una scelta di campione, non un dettaglio
# implementativo. La stessa costante e' usata in tutti e tre i punti che
# filtrano le aziende (fase 1, fase 2b, fase 4): finche' la sorgente e' una,
# le tre fasi non possono divergere.
with open(ROOT / "config" / "config.yaml") as f:
    CONFIG = yaml.safe_load(f)
ANNO_MIN_FONDAZIONE = int(CONFIG["first_year"])

# Da non confondere con la finestra temporale dei DATASET, che vive a valle in
# src/preprocessing.py e riguarda gli anni di calendario, non di fondazione.

print(f"anno di fondazione minimo: {ANNO_MIN_FONDAZIONE} (incluso), da config.yaml")
print("output di stadio         :", cfg.interim_dir)

# ── I tre interruttori della temporizzazione ──────────────────────────────
# Ogni interruttore sceglie, per un gruppo di attributi, fra il valore dell'anno
# della riga e la fotografia alla data di estrazione. Servono a produrre i due
# panel da confrontare, con e senza look-ahead, a parita' di tutto il resto:
#   TEMPORIZZA_PERSONE     -> esperienza e istruzione di team e CEO (2b.1bis, 5.6)
#   TEMPORIZZA_INVESTITORI -> dimensione e attivita' degli investitori (fase 4)
#   TEMPORIZZA_COMPETITOR  -> concorrenti e similarita' (fase 7)
# Un interruttore governa SOLO la temporizzazione del suo gruppo: tutto il resto
# della pipeline non cambia.
# TEMPORIZZA_INVESTITORI parte spento perche' la versione anno-per-anno vede solo
# i deal delle aziende dell'estrazione e quindi sottostima i fondi grandi: in
# media 5,1 investimenti ricostruiti contro i 20,5 dichiarati da Investor.csv.
TEMPORIZZA_PERSONE = True
TEMPORIZZA_INVESTITORI = False
TEMPORIZZA_COMPETITOR = True
print("temporizzazione          : persone", TEMPORIZZA_PERSONE,
      "\u00b7 investitori", TEMPORIZZA_INVESTITORI,
      "\u00b7 competitor", TEMPORIZZA_COMPETITOR)

anno di fondazione minimo: 2000 (incluso), da config.yaml
output di stadio         : data/interim_light
temporizzazione          : persone True · investitori False · competitor True


---
## Fase 1 — aziende e scheletro del panel

Da una riga per azienda a **una riga per azienda-anno**. È qui che nasce la
forma del panel.

Di 39 colonne di `Company.csv` ne servono **11**, e quattro di queste non
producono nessuna colonna finale: servono solo a calcolare `MaxYear`, cioè
dove finisce il panel di ogni azienda. Toglierne una sola costa righe —
misurato: senza `FiscalPeriod` se ne perdono 133.942.

In [2]:
# ── 1.1 · Company.csv, undici colonne su trentanove ───────────────────────
COLONNE_COMPANY = [
    "CompanyID",                    # chiave di tutto il panel
    "YearFounded",                  # anno zero di ogni azienda; e' una delle 53
    "HQCountry",                    # feature
    "PrimaryIndustrySector",        # feature
    "OwnershipStatus",              # NON e' una feature, ma serve in due punti scollegati:
                                    #   4  -> riparazione delle date dei deal
                                    #   5  -> i primi tre rami della cascata GrowthStage
    "OwnershipStatusDate",          # idem, piu' MaxYear
    # Le quattro che seguono, piu' FiscalPeriod, non producono nessuna colonna
    # del panel: entrano solo nel pmax che fa MaxYear.
    "CompanyFinancingStatusDate",
    "BusinessStatusDate",
    "FirstFinancingDate",
    "LastKnownValuationDate",
    "FiscalPeriod",                 # "TTM 2Q2019" -> FiscalDate, la sesta data di MaxYear
]

# read_raw legge tutto come stringa e non inferisce i tipi: CompanyID in certi
# file sembra numerico, e un cast implicito romperebbe i join in silenzio.
azienda = read_raw(cfg, "Company", COLONNE_COMPANY)

# as_na sostituisce con null i token che valgono "mancante": "", "NA", "N/A",
# "NULL" e "NaN". Va fatto PRIMA di qualunque altra cosa, perche' tutto il
# resto dipende da cosa conta come mancante.
azienda = as_na(azienda, R_NA_NAN)

print(f"{azienda.height:,} aziende x {azienda.width} colonne")

134,355 aziende x 11 colonne


In [3]:
# ── 1.2 · le date, e FiscalDate ───────────────────────────────────────────
# parse_date_r sceglie il formato in base alla lunghezza della stringa:
#   10 caratteri -> "%m/%d/%Y"
#    8 caratteri -> mese/giorno/anno a due cifre, con 00-24 nel 2000 e 25-99 nel 1900
#   qualunque altra lunghezza -> NA.
# Su questa estrazione tutte le date hanno 10 caratteri, quindi il secondo ramo
# non scatta mai: e' latente, non attivo.
azienda = azienda.with_columns(
    *[parse_date_r(pl.col(c)).alias(c) for c in COMPANY_DATE_COLUMNS],
    # YearFounded arriva come stringa; strict=False manda a null cio' che non e' un numero.
    pl.col("YearFounded").cast(pl.Int64, strict=False),
)

# FiscalPeriod ha la forma "TTM 2Q2019": si estrae il numero del trimestre...
trimestre = pl.col("FiscalPeriod").str.extract(r"TTM (\d)Q\d{4}", 1).cast(pl.Int64, strict=False)
# ...e l'anno, che sono le ultime quattro cifre della stringa.
anno_fiscale = pl.col("FiscalPeriod").str.slice(-4).cast(pl.Int64, strict=False)
# Il trimestre diventa il mese di chiusura (1Q->3, 2Q->6, 3Q->9, 4Q->12) e il
# giorno convenzionale e' sempre il 30.
azienda = azienda.with_columns(pl.date(anno_fiscale, trimestre * 3, 30).alias("FiscalDate"))

print(f"FiscalDate valorizzata su {azienda['FiscalDate'].is_not_null().sum():,} aziende")

FiscalDate valorizzata su 90,712 aziende


In [4]:
# ── 1.4 · MaxYear e lo scheletro ──────────────────────────────────────────
# MaxYear = l'anno piu' recente fra le sei date disponibili, cioe' l'ultimo
# anno in cui PitchBook ha un dato su quell'azienda. E' li' che finisce il suo panel.
DATE_MAXYEAR = [*COMPANY_DATE_COLUMNS, "FiscalDate"]

vita = (
    azienda
    # max_horizontal ignora i null: vale null solo se mancano tutte e sei le date.
    .with_columns(pl.max_horizontal([pl.col(c).dt.year() for c in DATE_MAXYEAR]).alias("MaxYear"))
    # Senza anno di fondazione o senza MaxYear l'azienda non entra nel panel:
    # e' questo il filtro che decide chi c'e' e chi no.
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
    .select("CompanyID", "YearFounded", "MaxYear")
)
print(f"aziende con anno di fondazione e MaxYear: {vita.height:,} su {azienda.height:,}")

# Il limite superiore e' il piu' grande fra MaxYear e l'anno di fondazione: il
# panel di un'azienda non puo' finire prima di iniziare. Serve perche' in 106
# aziende MaxYear PRECEDE la fondazione - tipicamente ri-registrazioni, 88 di
# loro hanno YearFounded nel 2024 o 2025 con dati di anni precedenti - e senza
# questo limite genererebbero anni anteriori alla nascita dell'azienda. Cosi'
# restano invece con una riga sola, l'anno zero.
limite = pl.max_horizontal("MaxYear", "YearFounded")

scheletro = (
    vita
    # r_seq restituisce una LISTA di anni per ogni riga...
    .with_columns(r_seq(pl.col("YearFounded"), limite).alias("Year_Delta"))
    # ...ed explode la apre in una riga per anno. empty_as_null tiene l'azienda
    # anche se la lista e' vuota, con un anno nullo.
    .explode("Year_Delta", empty_as_null=True)
    # Delta = eta' dell'azienda in quell'anno. Alla fase 5 diventera' Age.
    .with_columns((pl.col("Year_Delta") - pl.col("YearFounded")).alias("Delta"))
    .select("CompanyID", "YearFounded", "Year_Delta", "Delta")
)
print(f"scheletro: {scheletro.height:,} righe azienda-anno")
print(f"righe con Delta < 0 (deve essere 0): {scheletro.filter(pl.col('Delta') < 0).height}")

aziende con anno di fondazione e MaxYear: 126,817 su 134,355


scheletro: 1,309,093 righe azienda-anno
righe con Delta < 0 (deve essere 0): 0


In [5]:
# Dopo il blocco 1.4 lo scheletro è una griglia nuda: CompanyID, YearFounded, Year_Delta (anno di calendario), Delta (età). Sa chi esisteva e quando, ma niente su cosa gli succedeva.

# Company.csv ha una colonna OwnershipStatus — «Privately Held», «Acquired/Merged», «Out of Business» — che però non ha una dimensione temporale: è una riga per azienda, lo stato attuale. 
# Insieme c'è OwnershipStatusDate, cioè quando quello stato è stato raggiunto.

# ── 1.5 · l'unico join per anno che sopravvive ────────────────────────────
# Il notebook completo fa cinque join per anno: stato finanziario, stato di
# business, proprieta', ultima valutazione e bilanci. Qui ne serve UNO solo -
# la proprieta' - perche' e' l'unico che finisce (indirettamente) nelle 53:
# alimenta i primi tre rami della cascata GrowthStage.


anno_proprieta = (
    azienda
    .select(
        "CompanyID",
        pl.col("OwnershipStatusDate").dt.year().alias("_anno_own"),
        "OwnershipStatus",
    )
    # Una chiave di join nulla non deve agganciare niente. In polars i null non
    # si abbinano fra loro di default, quindi e' una sicurezza, non una necessita'.
    .drop_nulls("_anno_own")
)

# Left join: lo stato di proprieta' compare SOLO sulla riga dell'anno della sua
# data, non su tutti gli anni dell'azienda. Su tutte le altre righe resta nullo.
scheletro = scheletro.join(
    anno_proprieta,
    left_on=["CompanyID", "Year_Delta"],
    right_on=["CompanyID", "_anno_own"],
    how="left",
)
print(f"righe con OwnershipStatus valorizzato: {scheletro['OwnershipStatus'].is_not_null().sum():,}"
      f" su {scheletro.height:,}")

righe con OwnershipStatus valorizzato: 126,416 su 1,309,093


In [6]:
# ── 1.6 · il filtro sull'anno di fondazione, e la scrittura ───────────────
# db_master_1 ridotto a sei colonne: due feature e quattro che servono a valle.

# Mantiene solo le aziende fondate nell'anno minimo o dopo, e le scrive in un parquet che verra' usato in 2b e 4. Lo stesso filtro viene applicato allo scheletro,
#  perche' le aziende fondate prima non devono entrare nel panel finale.

db_master_1 = azienda.select(
    "CompanyID", "YearFounded", "HQCountry", "PrimaryIndustrySector",
    "OwnershipStatus", "OwnershipStatusDate",
).filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)

# Lo stesso filtro sullo scheletro, con la stessa costante.
scheletro = scheletro.filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)

db_master_1.write_parquet(cfg.interim("db_master_1.parquet"))
scheletro.write_parquet(cfg.interim("scheletro.parquet"))

print(f"db_master_1: {db_master_1.height:,} righe x {db_master_1.width} colonne  (attese 116.920)")
print(f"scheletro  : {scheletro.height:,} righe x {scheletro.width} colonne")

del azienda, vita, anno_proprieta, db_master_1
gc.collect()

db_master_1: 116,920 righe x 6 colonne  (attese 116.920)
scheletro  : 907,934 righe x 5 colonne


0

---
## Fase 2a — la tabella persona-azienda (`db3`)

Una riga per **coppia (azienda, persona)**, con gli attributi di quella persona
e la finestra di anni in cui era presente.

Entrano solo le aziende dello scheletro, e ogni coppia resta **una riga** anche
quando la persona ha avuto piu' ruoli (2a.1). Nessuna finestra va oltre l'ultimo
anno di vita dell'azienda (2a.10).

`db3` ha 10 colonne invece di 54: spariscono biografia, indirizzo, ateneo di
`Person.csv`, `RolesCount_Total`, `Is_Other` (sempre falsa) e i contatori
grezzi, che vengono consumati subito dall'indice di esperienza.

In [7]:
# ── 2a.1 · gli incarichi diventano una riga per coppia ────────────────────
# CompanyBoardTeamRelation ha una riga per INCARICO: la stessa persona nella
# stessa azienda compare piu' volte se ha piu' ruoli, o se lo stesso ruolo e'
# registrato due volte. Qui si tengono solo le aziende dello scheletro e si
# fondono in una riga per coppia (466.312 incarichi -> 465.861 coppie).
COLONNE_BOARD = [
    "CompanyID", "PersonID",     # la coppia, che e' anche la chiave della deduplica
    "PersonName",                # serve agli override Ph.D / JD / MD del blocco 2a.6
    "FullTitle",                 # serve a IsFounder (2a.7)
    "IsCurrent",                 # serve alle fusioni qui sotto
    "StartDate", "EndDate",      # la finestra di presenza
]
COPPIA = ["CompanyID", "PersonID"]

# LastUpdated serve solo alla prima fusione qui sotto, per scegliere fra due
# righe dello stesso incarico quale data e' piu' affidabile. db3 non la usa.
board = as_na(read_raw(cfg, "CompanyBoardTeamRelation", [*COLONNE_BOARD, "LastUpdated"]), R_NA_NAN)

# Solo le aziende dello scheletro: delle altre non entra niente nel panel.
# Dallo scheletro servono anche l'anno di fondazione (2a.9) e l'ULTIMO ANNO di
# vita dell'azienda (2a.10), cioe' dove finisce il suo panel.
vita_azienda = (
    pl.read_parquet(cfg.interim("scheletro.parquet"))
    .group_by("CompanyID")
    .agg(pl.col("YearFounded").first(), pl.col("Year_Delta").max().alias("UltimoAnno"))
)
print(f"righe grezze: {board.height:,}", end="   ")
board = board.join(vita_azienda.select("CompanyID"), on="CompanyID", how="semi")
print(f"nelle aziende dello scheletro: {board.height:,}   coppie distinte: {board.select(COPPIA).n_unique():,}")

# Righe dello STESSO INCARICO: stessa azienda, persona, nome e titolo. Nei dati
# sono sempre coppie (158 nelle aziende dello scheletro), che differiscono per
# IsCurrent e/o per le date.
# Si fondono in una riga sola, colonna per colonna:
#   IsCurrent - Yes se almeno una riga e' Yes.
#   EndDate   - nulla se la riga fusa e' Yes: "in carica" e "uscito il giorno X"
#               non possono stare insieme, e vince lo stato attuale.
#   date      - altrimenti vale la riga aggiornata piu' di recente (LastUpdated),
#               ma solo fra quelle che HANNO una data: una data vince sempre su
#               un null, anche se viene da un aggiornamento piu' vecchio.
#   spareggio - in circa meta' delle coppie LastUpdated e' identico (138 su 281
#               sull'estrazione intera): fra le date
#               aggiornate lo stesso giorno si prende l'intervallo piu' ampio,
#               StartDate piu' vecchia ed EndDate piu' recente.
# Le date sono ancora testo: si confrontano come date (parse_date_r) e si
# riscrivono come mm/dd/yyyy, che 2a.2 rilegge identiche. Le righe senza una
# seconda riga dello stesso incarico restano esattamente com'erano.
STESSO_INCARICO = ["CompanyID", "PersonID", "PersonName", "FullTitle"]
aggiornata = parse_date_r(pl.col("LastUpdated"))
in_carica = (pl.col("IsCurrent") == "Yes").any()
multiple = pl.len() > 1


def data_scelta(colonna: str, spareggio: str) -> pl.Expr:
    """La data della riga aggiornata piu' di recente fra quelle che ne hanno una;
    a parita' di LastUpdated, la min (StartDate) o la max (EndDate)."""
    data = parse_date_r(pl.col(colonna))
    ha_data = data.is_not_null()
    candidate = data.filter(ha_data & (aggiornata == aggiornata.filter(ha_data).max()))
    scelta = candidate.min() if spareggio == "min" else candidate.max()
    return scelta.dt.strftime("%m/%d/%Y")


prima = board.height
board = (
    board.with_row_index("_riga")
    .group_by(STESSO_INCARICO)
    .agg(
        # _riga tiene l'incarico nella posizione della sua prima occorrenza.
        pl.col("_riga").min(),
        pl.when(multiple & in_carica).then(pl.lit("Yes"))
        .otherwise(pl.col("IsCurrent").drop_nulls().first())
        .alias("IsCurrent"),
        pl.when(multiple).then(data_scelta("StartDate", "min"))
        .otherwise(pl.col("StartDate").first())
        .alias("StartDate"),
        pl.when(multiple & in_carica).then(pl.lit(None, dtype=pl.String))
        .when(multiple).then(data_scelta("EndDate", "max"))
        .otherwise(pl.col("EndDate").first())
        .alias("EndDate"),
    )
    .sort("_riga")
    .select(COLONNE_BOARD)          # LastUpdated ha finito il suo lavoro
)
print(f"righe fuse perche' stesso incarico: {prima - board.height:,}")

# Ruoli DIVERSI della stessa coppia (es. "Founder & CEO" 1992-2013 e "Chairman"
# 1992-2015). Le variabili di team sono per persona, non per ruolo: conta solo in
# quali anni la persona c'era, quindi si fonde l'UNIONE dei periodi.
#   StartDate - nulla se almeno un ruolo non ha inizio, altrimenti la piu' vecchia.
#               Un ruolo senza inizio parte dalla fondazione (2a.9), e l'unione con
#               lui. E' l'opposto del blocco precedente, e non per sbaglio: li' le
#               righe erano lo STESSO incarico e il null un'informazione mancante;
#               qui sono ruoli diversi.
#   EndDate   - nulla se almeno un ruolo e' Yes o non ha fine, altrimenti la piu'
#               recente: meglio contare una persona un anno di troppo che perderla.
#   IsCurrent - Yes se almeno un ruolo e' Yes.
#   FullTitle - titoli distinti uniti con ", ", cosi' chi e' founder in uno dei
#               ruoli resta founder anche dopo la fusione.
# Limite: se fra due ruoli c'e' un buco, la persona risulta presente anche negli
# anni scoperti. Misurato dopo il taglio di 2a.10: 2 persone e 5 anni-azienda
# (Johan Englund 2019-2022, Nicholas Beal 2024). Tenere i ruoli separati fino
# all'espansione li eviterebbe, al costo di rendere unica la persona per anno in
# 2b.2 e il CEO in 5.6: per 5 anni-azienda non vale.
# Una data non leggibile conta come nulla, come diventera' comunque in 2a.2.
# Le coppie con una riga sola restano esattamente com'erano.
inizio, fine = parse_date_r(pl.col("StartDate")), parse_date_r(pl.col("EndDate"))
prima = board.height
board = (
    board.with_row_index("_riga")
    .group_by(COPPIA)
    .agg(
        pl.col("_riga").min(),
        pl.col("PersonName").first(),
        pl.when(pl.col("FullTitle").is_not_null().any())
        .then(pl.col("FullTitle").drop_nulls().unique(maintain_order=True).str.join(", "))
        .alias("FullTitle"),
        pl.when(multiple & in_carica).then(pl.lit("Yes"))
        .otherwise(pl.col("IsCurrent").drop_nulls().first())
        .alias("IsCurrent"),
        pl.when(multiple & inizio.is_null().any()).then(pl.lit(None, dtype=pl.String))
        .when(multiple).then(inizio.min().dt.strftime("%m/%d/%Y"))
        .otherwise(pl.col("StartDate").first())
        .alias("StartDate"),
        pl.when(multiple & (in_carica | fine.is_null().any())).then(pl.lit(None, dtype=pl.String))
        .when(multiple).then(fine.max().dt.strftime("%m/%d/%Y"))
        .otherwise(pl.col("EndDate").first())
        .alias("EndDate"),
    )
    .sort("_riga")
    .select(COLONNE_BOARD)
)
print(f"righe fuse perche' ruoli diversi della stessa coppia: {prima - board.height:,}")

assert board.height == board.select(COPPIA).n_unique(), "restano coppie su piu' righe"
print(f"dopo la deduplica: {board.height:,} righe, una per coppia")

# ── I ruoli da CEO, con le loro finestre ──────────────────────────────────
# Serve al blocco 5.6 per tappare il buco dei CEO negli anni prima del primo
# round. Si costruisce QUI perche' piu' avanti StartDate ed EndDate spariscono
# (2a.11 le converte in DeltaStart/DeltaEnd) e il titolo non basterebbe.
#
# La fusione dei ruoli qui sopra non e' un problema: la tabella grezza ha in
# pratica una riga per coppia (535.568 righe su 534.851 coppie), quindi per il
# 99,9% delle persone la riga fusa E' la riga originale.
#
# Attenzione a cosa significano queste date, perche' decide tutto: la StartDate
# e' l'inizio del RAPPORTO con l'azienda, non del ruolo da CEO. *Verificato: le
# coppie con sia una riga CEO sia una non-CEO sono 116 su 534.851, e fra quelle
# datate il 57% ha la stessa data.* Quindi da sola non dice quando uno e'
# diventato CEO: e' 5.6 a incrociarla con i deal per stabilirlo.
_titolo = pl.col("FullTitle").fill_null("")
_senza_assistenti = _titolo.str.replace_all(r"(?i)founders?'?s?'? associate", "")
_sv = parse_date_r(pl.col("StartDate")).dt.year()
_ev = parse_date_r(pl.col("EndDate")).dt.year()

ruoli_ceo = (
    board.filter(_titolo.str.contains(r"(?i)\bceo\b|chief executive"))
    .join(vita_azienda, on="CompanyID", how="inner")
    .with_columns(
        _sv.alias("sv"),                                   # anno d'inizio VERO, se c'e'
        _senza_assistenti.str.contains(r"(?i)founde|founding").alias("is_founder"),
    )
    .with_columns(
        pl.max_horizontal(pl.coalesce("sv", "YearFounded"), pl.col("YearFounded")).alias("da"),
        pl.min_horizontal(pl.coalesce(_ev, pl.col("UltimoAnno")), pl.col("UltimoAnno")).alias("a"),
    )
    .filter(pl.col("da") <= pl.col("a"))
    .select("CompanyID", "PersonID", "da", "a", "sv", "is_founder")
)
ruoli_ceo.write_parquet(cfg.interim("ruoli_ceo.parquet"))
print(f"ruoli da CEO nel board: {ruoli_ceo.height:,}  "
      f"(founder+CEO: {ruoli_ceo['is_founder'].sum():,}, con data vera: {ruoli_ceo['sv'].is_not_null().sum():,})")


righe grezze: 535,568   

nelle aziende dello scheletro: 466,312   coppie distinte: 465,861


righe fuse perche' stesso incarico: 158


righe fuse perche' ruoli diversi della stessa coppia: 293
dopo la deduplica: 465,861 righe, una per coppia


ruoli da CEO nel board: 94,464  (founder+CEO: 67,410, con data vera: 79,873)


In [8]:
# ── 2a.2 · le date della permanenza ───────────────────────────────────────
db3 = (
    board
    .with_columns(parse_date_r(pl.col(c)).alias(c) for c in ("StartDate", "EndDate"))
    # Questo ordinamento conta anche a valle: decide in che ordine gli atenei
    # finiranno concatenati in Institute alla fase 2b. nulls_last manda in coda
    # le righe senza data d'inizio.
    .sort(["CompanyID", "StartDate"], nulls_last=True)
)
del board
gc.collect()
print(f"StartDate valorizzate: {db3['StartDate'].is_not_null().sum():,} su {db3.height:,}")

StartDate valorizzate: 324,481 su 465,861


In [9]:
# ── 2a.3 · il genere ──────────────────────────────────────────────────────
# Da Person.csv serve solo il genere. I conteggi di ruoli che alimentano
# WorkExperienceIndex non si leggono da qui: Person.csv li espone come totali
# alla data di estrazione, mentre servono anno per anno. Si ricostruiscono in
# 2a.3bis, e l'indice si calcola dopo l'espansione, in 2b.1bis.
persona = as_na(read_raw(cfg, "Person", ["PersonID", "Gender"]), R_NA_NAN)
db3 = db3.join(persona, on="PersonID", how="left")
del persona
gc.collect()
print(f"righe senza genere: {db3['Gender'].is_null().sum():,} su {db3.height:,}")


righe senza genere: 2,683 su 465,861


In [10]:
# ── 2a.3bis · l'esperienza anno per anno: gli eventi e i conteggi cumulati ──
# L'esperienza di una persona, anno per anno. Person.csv espone otto contatori
# di ruoli, ma sono totali alla data di estrazione: non dicono quanti ruoli
# quella persona avesse nel 2012. Ognuno si ricostruisce da una tabella di
# dettaglio con una riga per ruolo:
#   posizioni   CurrentPositionsCount + FormerPositionsCount         PersonPositionRelation
#   seggi       CurrentBoardSeatsCount + FormerBoardSeatsCount       PersonBoardSeatRelation
#   altri ruoli CurrentAdvisoryRolesCount + FormerAdvisoryRolesCount  PersonAdvisoryRelation
#               AffiliatedDealsCount                                 PersonAffiliatedDealRelation
#               NumberOfAffiliatedFunds                              PersonAffiliatedFundRelation
# Ogni ruolo diventa un EVENTO con un anno, e l'esperienza all'anno Y e' il numero
# di ruoli iniziati entro Y, finiti o no: la EndDate non serve.
#
# L'anno di un ruolo, in ordine:
#   1. la data vera (StartDate; per i deal la DealDate);
#   2. se manca, l'anno di fondazione dell'entita' in cui si svolge (Company.csv
#      o Investor.csv): non e' la data vera ma un limite inferiore, un ruolo non
#      puo' iniziare prima che l'entita' esista;
#   3. se manca anche quello, il PRIMO ANNO NOTO della persona: il piu' antico
#      fra gli anni risolti ai punti 1 e 2 sui suoi altri ruoli. Le entita' che
#      finiscono qui sono aziende FUORI dalla nostra estrazione (il 93,1%: non
#      stanno ne' in Company.csv ne' in Investor.csv) oppure entita'-persona
#      come gli angel, quindi non esiste una fondazione a cui ancorarle. Nessuna
#      di loro e' un'azienda del panel;
#   4. se la persona non ha nemmeno un altro ruolo datato, anno 0 ("conta da
#      sempre"), come faceva il contatore. Misurato: non capita mai, ma il ramo
#      resta perche' e' la sola uscita possibile.
# Per i fondi la data vera non esiste: l'affiliazione non puo' precedere ne' la
# nascita del fondo (Vintage) ne' l'ingresso della persona nella societa'
# d'investimento che lo gestisce, quindi vale la piu' recente delle due.
# Qui la tabella si costruisce; la cella 2a.3ter la confronta con Person.csv, e
# WorkExperienceIndex la usa dopo l'espansione, in 2b.1bis.
persone = db3.select("PersonID").unique()
anno_di = lambda colonna: parse_date_r(pl.col(colonna)).dt.year()


def leggi(tabella: str, colonne: list[str]) -> pl.DataFrame:
    """Una tabella delle persone, solo per le persone di db3."""
    return as_na(read_raw(cfg, tabella, colonne), R_NA_NAN).join(persone, on="PersonID", how="semi")


# Anno di fondazione di ogni entita': tutte le aziende, non solo quelle del
# panel, e le societa' d'investimento. 4.869 ID stanno in entrambe le tabelle:
# sono la stessa entita' (un'azienda che investe) e, dove l'anno c'e', e' uguale
# in tutte e due. Si tiene quello di Company.csv.
fondazioni = pl.concat([
    as_na(read_raw(cfg, "Company", ["CompanyID", "YearFounded"]), R_NA_NAN)
    .select(pl.col("CompanyID").alias("Entita"), pl.col("YearFounded").cast(pl.Int64, strict=False).alias("Fondazione")),
    as_na(read_raw(cfg, "Investor", ["InvestorID", "YearFounded"]), R_NA_NAN)
    .select(pl.col("InvestorID").alias("Entita"), pl.col("YearFounded").cast(pl.Int64, strict=False).alias("Fondazione")),
]).drop_nulls().unique("Entita", keep="first", maintain_order=True)

posizioni = leggi("PersonPositionRelation", ["PersonID", "EntityID", "StartDate"])
seggi = leggi("PersonBoardSeatRelation", ["PersonID", "CompanyID", "StartDate"])
advisory = leggi("PersonAdvisoryRelation", ["PersonID", "EntityID", "StartDate"])
deal = leggi("PersonAffiliatedDealRelation", ["PersonID", "CompanyID", "DealDate"])
fondi = leggi("PersonAffiliatedFundRelation", ["PersonID", "FundID", "InvestorID"])

# Per i fondi: l'ingresso della persona nella societa' d'investimento e' la sua
# posizione PIU' VECCHIA li', contando solo le StartDate vere.
ingresso = (
    posizioni.with_columns(anno_di("StartDate").alias("_ingresso"))
    .group_by("PersonID", "EntityID").agg(pl.col("_ingresso").min())
)
fondi_datati = (
    fondi.join(as_na(read_raw(cfg, "Fund", ["FundID", "Vintage"]), R_NA_NAN)
               .with_columns(pl.col("Vintage").cast(pl.Int64, strict=False)), on="FundID", how="left")
    .join(ingresso, left_on=["PersonID", "InvestorID"], right_on=["PersonID", "EntityID"], how="left")
    # max_horizontal ignora i null: con una sola delle due date prende quella.
    .with_columns(pl.max_horizontal("Vintage", "_ingresso").alias("_affiliazione"))
)

# fonte: (righe lette, colonna dell'entita', anno vero, gruppo, contatori di Person.csv)
FONTI = {
    "posizioni": (posizioni, "EntityID", anno_di("StartDate"), "Posizioni",
                  ["CurrentPositionsCount", "FormerPositionsCount"]),
    "seggi": (seggi, "CompanyID", anno_di("StartDate"), "Seggi",
              ["CurrentBoardSeatsCount", "FormerBoardSeatsCount"]),
    "advisory": (advisory, "EntityID", anno_di("StartDate"), "AltriRuoli",
                 ["CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount"]),
    "deal": (deal, "CompanyID", anno_di("DealDate"), "AltriRuoli", ["AffiliatedDealsCount"]),
    "fondi": (fondi_datati, "InvestorID", pl.col("_affiliazione"), "AltriRuoli", ["NumberOfAffiliatedFunds"]),
}


def eventi(nome: str, righe: pl.DataFrame, entita: str, anno: pl.Expr, gruppo: str) -> pl.DataFrame:
    """Un evento per ruolo: data vera, altrimenti fondazione dell'entita'.

    Chi non ha nessuna delle due esce con anno 0 e origine "da risolvere": lo
    sistema il blocco qui sotto, che ha bisogno di TUTTI gli eventi della
    persona e quindi non puo' girare qui dentro.
    """
    return (
        righe.join(fondazioni, left_on=entita, right_on="Entita", how="left")
        .select("PersonID",
                pl.coalesce(anno, pl.col("Fondazione"), pl.lit(0)).cast(pl.Int64).alias("anno"),
                pl.lit(gruppo).alias("gruppo"),
                pl.lit(nome).alias("fonte"),
                pl.when(anno.is_not_null()).then(pl.lit("data"))
                .when(pl.col("Fondazione").is_not_null()).then(pl.lit("fondazione"))
                .otherwise(pl.lit("sempre")).alias("origine"))
    )


tutti = pl.concat([eventi(nome, *f[:4]) for nome, f in FONTI.items()])

# Punto 3 della cascata. Un ruolo senza data e senza fondazione dell'entita'
# valeva "da sempre": contava in OGNI anno del panel, anche vent'anni prima che
# di quella persona si sapesse qualcosa. Ora parte dal suo primo anno noto, che
# e' l'unico limite inferiore difendibile. Il join e' per PersonID su una tabella
# con una riga per persona, quindi non puo' moltiplicare le righe: lo garantisce
# il Controllo 1 qui sotto.
primo_anno = (
    tutti.filter(pl.col("origine") != "sempre")
    .group_by("PersonID").agg(pl.col("anno").min().alias("_primo"))
)
tutti = (
    tutti.join(primo_anno, on="PersonID", how="left")
    .with_columns(
        pl.when(pl.col("origine") != "sempre").then(pl.col("origine"))
        .when(pl.col("_primo").is_not_null()).then(pl.lit("primo anno"))
        .otherwise(pl.lit("sempre")).alias("origine"),
        pl.when(pl.col("origine") == "sempre")
        .then(pl.col("_primo").fill_null(0))
        .otherwise(pl.col("anno")).cast(pl.Int64).alias("anno"),
    )
    .drop("_primo")
)
del primo_anno

print(f"eventi: {tutti.height:,} per {tutti['PersonID'].n_unique():,} persone")
print(tutti.group_by("gruppo", "origine").len().sort("gruppo", "origine"))

# ── Controllo 1, sul codice: ogni riga letta produce esattamente un evento. Se
# un join duplica o perde righe, qui si vede.
for nome, (righe, *_) in FONTI.items():
    prodotti = tutti.filter(pl.col("fonte") == nome).height
    assert prodotti == righe.height, f"{nome}: {righe.height:,} righe lette ma {prodotti:,} eventi"

# I conteggi CUMULATI: una riga per (persona, anno) in cui cambia qualcosa, con
# quanti ruoli di ciascun gruppo sono iniziati fino a quell'anno compreso.
GRUPPI = ["Posizioni", "Seggi", "AltriRuoli"]
esperienza = (
    tutti.group_by("PersonID", "anno")
    .agg(*[(pl.col("gruppo") == g).sum().cast(pl.Int64).alias(g) for g in GRUPPI])
    .sort("PersonID", "anno")
    .with_columns(*[pl.col(g).cum_sum().over("PersonID") for g in GRUPPI])
)

esperienza.write_parquet(cfg.interim("esperienza_persona_anno.parquet"))
print(f"esperienza_persona_anno: {esperienza.height:,} righe x {esperienza.width} colonne")
del persone, fondazioni, posizioni, seggi, advisory, deal, fondi, ingresso, fondi_datati
# del FONTI, tutti, esperienza
gc.collect()


eventi: 918,698 per 404,468 persone
shape: (9, 3)
┌────────────┬────────────┬────────┐
│ gruppo     ┆ origine    ┆ len    │
│ ---        ┆ ---        ┆ ---    │
│ str        ┆ str        ┆ u32    │
╞════════════╪════════════╪════════╡
│ AltriRuoli ┆ data       ┆ 220567 │
│ AltriRuoli ┆ fondazione ┆ 10059  │
│ AltriRuoli ┆ primo anno ┆ 4566   │
│ Posizioni  ┆ data       ┆ 347122 │
│ Posizioni  ┆ fondazione ┆ 110062 │
│ Posizioni  ┆ primo anno ┆ 18788  │
│ Seggi      ┆ data       ┆ 142653 │
│ Seggi      ┆ fondazione ┆ 52288  │
│ Seggi      ┆ primo anno ┆ 12593  │
└────────────┴────────────┴────────┘


esperienza_persona_anno: 639,681 righe x 5 colonne


0

In [11]:
# ── 2a.3ter · controllo: Person.csv contro le tabelle dei ruoli ────────────
# Due confronti, per ogni persona di esperienza_persona_anno.
#   1. CONTATORE PER CONTATORE: ciascuno degli 8 contatori di Person.csv contro
#      le righe della sua tabella. Per posizioni, seggi e advisor gli attuali si
#      contano sulle righe con IsCurrent = "Yes" e i passati su "No"; per deal e
#      fondi su tutte le righe.
#   2. PER GRUPPO: la somma dei contatori, come la fa 2a.3, contro l'ultima riga
#      di esperienza, che contiene tutti i ruoli di qualunque anno.
# Un contatore vuoto in Person.csv vale 0, come in 2a.3. Il riepilogo separa le
# differenze nate da un contatore vuoto da quelle con un contatore valorizzato.
CONTATORI = {  # contatore: (tabella, valore di IsCurrent da contare; None = tutte le righe)
    "CurrentPositionsCount": ("PersonPositionRelation", "Yes"),
    "FormerPositionsCount": ("PersonPositionRelation", "No"),
    "CurrentBoardSeatsCount": ("PersonBoardSeatRelation", "Yes"),
    "FormerBoardSeatsCount": ("PersonBoardSeatRelation", "No"),
    "CurrentAdvisoryRolesCount": ("PersonAdvisoryRelation", "Yes"),
    "FormerAdvisoryRolesCount": ("PersonAdvisoryRelation", "No"),
    "AffiliatedDealsCount": ("PersonAffiliatedDealRelation", None),
    "NumberOfAffiliatedFunds": ("PersonAffiliatedFundRelation", None),
}
GRUPPO_DI = {
    "Posizioni": ["CurrentPositionsCount", "FormerPositionsCount"],
    "Seggi": ["CurrentBoardSeatsCount", "FormerBoardSeatsCount"],
    "AltriRuoli": ["CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount",
                   "AffiliatedDealsCount", "NumberOfAffiliatedFunds"],
}

ultima = (
    pl.read_parquet(cfg.interim("esperienza_persona_anno.parquet"))
    .sort("PersonID", "anno")
    .group_by("PersonID")
    .agg(pl.col(*GRUPPO_DI).last())
)
persone = ultima.select("PersonID")

# Le righe di ogni tabella, contate per persona e per contatore.
tabelle = {}
for tabella, stato in CONTATORI.values():
    if tabella not in tabelle:
        colonne = ["PersonID", "IsCurrent"] if stato is not None else ["PersonID"]
        tabelle[tabella] = as_na(read_raw(cfg, tabella, colonne), R_NA_NAN).join(persone, on="PersonID", how="semi")
righe = persone
for contatore, (tabella, stato) in CONTATORI.items():
    df = tabelle[tabella] if stato is None else tabelle[tabella].filter(pl.col("IsCurrent") == stato)
    righe = righe.join(df.group_by("PersonID").agg(pl.len().cast(pl.Int64).alias(f"righe_{contatore}")),
                       on="PersonID", how="left")
righe = righe.with_columns(pl.col("^righe_.*$").fill_null(0))

person = as_na(read_raw(cfg, "Person", ["PersonID", "FullName", *CONTATORI]), R_NA_NAN).with_columns(
    to_num(c) for c in CONTATORI
)
confronto = persone.join(person, on="PersonID", how="left").join(righe, on="PersonID").join(ultima, on="PersonID")
assenti = confronto["FullName"].null_count()
if assenti:
    print(f"ATTENZIONE: {assenti:,} persone di esperienza non sono in Person.csv")

# ── 1. contatore per contatore ────────────────────────────────────────────
riepilogo = []
for contatore in CONTATORI:
    vuoto, n = pl.col(contatore).is_null(), pl.col(f"righe_{contatore}")
    diverso = pl.col(contatore).fill_null(0) != n
    riepilogo.append(confronto.select(
        pl.lit(contatore).alias("contatore"),
        (~diverso).sum().alias("uguali"),
        (vuoto & diverso).sum().alias("diversi, contatore vuoto"),
        (~vuoto & diverso).sum().alias("diversi, contatore valorizzato"),
        (vuoto & (n == 0)).sum().alias("vuoti con 0 righe"),
    ))
riepilogo = pl.concat(riepilogo)
with pl.Config(tbl_cols=6, tbl_width_chars=200, tbl_rows=10):
    print(riepilogo)

diversi_contatore = riepilogo.select(pl.col("^diversi.*$")).sum().sum_horizontal().item()
if diversi_contatore == 0:
    print("Contatori: identici per tutte le persone")
else:
    print(f"\nATTENZIONE: {diversi_contatore} contatori diversi dalle righe delle loro tabelle")
    for contatore in CONTATORI:
        qui = confronto.filter(pl.col(contatore).fill_null(0) != pl.col(f"righe_{contatore}"))
        if qui.height:
            with pl.Config(tbl_cols=6, tbl_width_chars=200, fmt_str_lengths=30):
                print(f"\n{contatore}:")
                print(qui.select("PersonID", "FullName", pl.col(contatore).alias("Person.csv"),
                                 pl.col(f"righe_{contatore}").alias("righe nella tabella")))

# ── 2. per gruppo, contro l'ultima riga di esperienza ─────────────────────
# Le righe delle tabelle sommate per gruppo devono dare ESATTAMENTE l'ultima riga:
# e' la stessa informazione, quindi una differenza qui e' un errore di 2a.3bis.
for g, contatori in GRUPPO_DI.items():
    sbagliate = confronto.filter(pl.sum_horizontal([f"righe_{c}" for c in contatori]) != pl.col(g)).height
    assert sbagliate == 0, f"{g}: {sbagliate:,} persone con l'ultima riga diversa dalle righe delle tabelle"
diversi_gruppo = confronto.filter(pl.any_horizontal([
    pl.sum_horizontal([pl.col(c).fill_null(0) for c in contatori]) != pl.col(g) for g, contatori in GRUPPO_DI.items()
])).height
print(f"\nPer gruppo: l'ultima riga coincide con le tabelle per tutte le {confronto.height:,} persone; "
      f"con Person.csv differisce per {diversi_gruppo} persone (le stesse dei contatori qui sopra)")
del ultima, persone, tabelle, righe, person, confronto, riepilogo
gc.collect()


shape: (8, 5)
┌───────────────────────────┬────────┬──────────────────────────┬────────────────────────────────┬───────────────────┐
│ contatore                 ┆ uguali ┆ diversi, contatore vuoto ┆ diversi, contatore valorizzato ┆ vuoti con 0 righe │
│ ---                       ┆ ---    ┆ ---                      ┆ ---                            ┆ ---               │
│ str                       ┆ u32    ┆ u32                      ┆ u32                            ┆ u32               │
╞═══════════════════════════╪════════╪══════════════════════════╪════════════════════════════════╪═══════════════════╡
│ CurrentPositionsCount     ┆ 404468 ┆ 0                        ┆ 0                              ┆ 176276            │
│ FormerPositionsCount      ┆ 404468 ┆ 0                        ┆ 0                              ┆ 222948            │
│ CurrentBoardSeatsCount    ┆ 404467 ┆ 1                        ┆ 0                              ┆ 321432            │
│ FormerBoardSeatsCount     ┆ 4044

0

In [12]:
# ── 2a.4 · il titolo di studio e il campo di studi ────────────────────────
# Due catene di grepl a CORTO CIRCUITO: il primo che matcha vince e gli altri
# non vengono nemmeno valutati. L'ordine delle regole e' quindi parte della
# logica: "MD" compare nella prima regola, quindi un MD non arriva mai a
# "Master's" anche se la seconda regola contiene "Master".
REGOLE_TITOLO = [
    (r"PhD|Doctor|MD|PsyD|DPhil|DC|DDS|DPT|OD|JD", "PhD/Doctorate"),
    (r"MBA|LLM|Master|MSc|MPhil|MPA|MFA|MEng|MAcc|Graduate", "Master's"),
    (r"Bachelor|BSc|BEng|Laurea|BS|BFA|BCom|degree|Business Program|Undergrad", "Bachelor's"),
    (
        r"Certified|Certificat|A Levels|A-Levels|Diplom|DEA|DESS|Dipl\.-Ing|"
        r"Executive Development Program|Executive Education|Executive Education program|"
        r"Executive Program|First Legal State Exam|Legal Practice Course|Vordiplom",
        "Diploma/Certificate",
    ),
]
# La gerarchia: Highest_Degree e' l'indice 1-based del livello piu' alto raggiunto.
# Il ramo finale "Other" cattura anche un titolo mancante, quindi DegreeLevel non
# e' mai nullo: chi non dichiara nulla prende 1, non NA.
GERARCHIA = ["Other", "Diploma/Certificate", "Bachelor's", "Master's", "PhD/Doctorate"]

REGOLE_AREA = [
    (r"business|management|bank|invest|financ|marketing|real estate|account|"
     r"Entrepreneur|commerce|econom|actuarial science|private equity", "Economics"),
    (r"engineer|civil|electric|mechanic|electronic|Operations Research|material|"
     r"logistic|Engeneering", "Engineering"),
    (r"statistic|machine learning|Natural Language|robot|technolog|comput|data|"
     r"informatic|Artificial Intelligence|information science|information systems|"
     r"data science|softwar", "IT and Computer Science"),
    (r"law|tax|justice|forensic|legal|jurisprudence|intellectual property", "Law"),
    (r"medic|nursing|pharmac|health|immunolog|neuroscien|genetic|physio", "Health and Medicine"),
    (r"social science|strateg|sociology|psychology|anthropology|international relations|"
     r"polit|government|geography|polic|international|foreign service|social studies|"
     r"criminology|cognitive science|public affairs|urban planning|social work|"
     r"human resource|leadership|foreign", "Social Sciences"),
    (r"natural|biolog|chemistry|physic|environmental science|math|geology|life science|"
     r"zoology|agriculture", "Natural Sciences"),
    (r"humanit|literature|histor|philosoph|language|linguist|english|spanis|american|"
     r"religion|classic|french|theolog|cultural studies|europe|arts|design|music|"
     r"architecture|journalism|media|public relations|advertising|communic|education|"
     r"early childhood|special education|administration", "Humanities and Arts"),
]

studi = as_na(
    read_raw(cfg, "PersonEducationRelation",
             ["PersonID", "Degree", "Major_Concentration", "GraduatingYear", "Institute"]),
    R_NA_NAN,
)
# r_case_when applica le regole in ordine; chi non matcha niente finisce in "Other".
studi = studi.with_columns(
    r_case_when(REGOLE_TITOLO, pl.col("Degree"), pl.lit("Other")).alias("DegreeLevel")
)

titolo, area = pl.col("Degree"), pl.col("Major_Concentration")
studi = studi.with_columns(
    # Se l'area di studi manca, si prova a dedurla dal nome del titolo.
    # fill_null(False) perche' str.contains su un valore nullo da' null, e
    # pl.when tratterebbe quel null come "non matcha" - qui e' quello che vogliamo,
    # ma scriverlo esplicito evita di doverci ripensare.
    pl.when(area.is_null() & titolo.str.contains("(?i)law").fill_null(False)).then(pl.lit("Law"))
    .when(area.is_null() & titolo.str.contains("(?i)MBA").fill_null(False)).then(pl.lit("Business"))
    .when(area.is_null() & titolo.str.contains("(?i)Medicine").fill_null(False)).then(pl.lit("Medicine"))
    .otherwise(area)
    .alias("Major_Concentration")
)
studi = studi.with_columns(
    # Se l'area e' ancora nulla, Field resta nulla. Non "Other": nulla.
    # La differenza conta, perche' i flag Is_* qui sotto
    # distinguono "nessun titolo classificabile" da "titolo di area Other".
    pl.when(pl.col("Major_Concentration").is_null())
    .then(None)
    .otherwise(r_case_when(REGOLE_AREA, pl.col("Major_Concentration"), pl.lit("Other")))
    .alias("Field")
)
print(studi["Field"].value_counts(sort=True))

shape: (10, 2)
┌─────────────────────────┬────────┐
│ Field                   ┆ count  │
│ ---                     ┆ ---    │
│ str                     ┆ u32    │
╞═════════════════════════╪════════╡
│ Economics               ┆ 488278 │
│ null                    ┆ 238415 │
│ Law                     ┆ 153996 │
│ Engineering             ┆ 99971  │
│ Humanities and Arts     ┆ 53061  │
│ Natural Sciences        ┆ 52460  │
│ Social Sciences         ┆ 48137  │
│ Other                   ┆ 45856  │
│ IT and Computer Science ┆ 44208  │
│ Health and Medicine     ┆ 21672  │
└─────────────────────────┴────────┘


In [13]:
# ── 2a.5 · l'istruzione anno per anno ─────────────────────────────────────
# L'istruzione di una persona, anno per anno: per ogni anno contano solo i
# titoli conseguiti ENTRO quell'anno, altrimenti un MBA preso nel 2018 alzerebbe
# il titolo di studio gia' nella riga del 2010.
#   - L'anno di un titolo e' GraduatingYear. ASSUNZIONE: se manca (35% dei
#     titoli) vale anno 0, cioe' il titolo conta in ogni anno del panel.
#   - Per ogni persona i "punti" sono gli anni dei suoi titoli: in ciascuno si
#     prendono i titoli con anno <= punto e si aggregano. Fra un punto e l'altro
#     non cambia niente, e il join per anno di 2b.1bis prende il piu' recente.
#   - I pochi titoli con un anno futuro (2026, 2027, 2033) non contano in nessun
#     anno del panel.
AREE = {
    "Is_Eco": "Economics",
    "Is_Eng": "Engineering",
    "Is_Med": "Health and Medicine",
    "Is_Hum": "Humanities and Arts",
    "Is_IT": "IT and Computer Science",
    "Is_Law": "Law",
    "Is_NS": "Natural Sciences",
    "Is_SS": "Social Sciences",
}

# Da "Master's" all'indice 4. default=None perche' un valore fuori gerarchia
# sarebbe un errore, non uno zero.
indice_titolo = (
    pl.col("DegreeLevel")
    .replace_strict({nome: i + 1 for i, nome in enumerate(GERARCHIA)}, default=None)
    .cast(pl.Int64)
)
# Guardia: se la persona non ha nessun campo di studi classificabile i flag
# restano NULLI, non falsi. "Non lo sappiamo" e "non e' di quell'area" sono
# cose diverse.
ha_campo = pl.col("Field").is_not_null().sum() > 0

# L'aggregazione, usata sia qui sia dal controllo 2a.5bis.
AGGREGAZIONE = [
    *[pl.when(ha_campo).then(pl.col("Field").eq(valore).any()).otherwise(None).alias(flag)
      for flag, valore in AREE.items()],
    # Earliest_Year: il primo anno di laurea fra i titoli gia' conseguiti.
    pl.col("GraduatingYear").cast(pl.Float64, strict=False).min().alias("Earliest_Year"),
    indice_titolo.max().alias("Highest_Degree"),
    # Gli atenei si concatenano con "; " scartando i mancanti.
    pl.when(pl.col("Institute").is_not_null().sum() > 0)
    .then(pl.col("Institute").drop_nulls().str.join("; "))
    .otherwise(None)
    .alias("Institute"),
]

titoli = (
    studi.join(db3.select("PersonID").unique(), on="PersonID", how="semi")
    # _ordine conserva l'ordine delle righe, che decide quello degli atenei in Institute.
    .with_row_index("_ordine")
    .with_columns(pl.col("GraduatingYear").cast(pl.Int64, strict=False).fill_null(0).alias("anno_titolo"))
)
punti = titoli.select("PersonID", pl.col("anno_titolo").alias("anno")).unique()
istruzione = (
    punti.join(titoli, on="PersonID")
    .filter(pl.col("anno_titolo") <= pl.col("anno"))
    .sort("_ordine")
    .group_by("PersonID", "anno", maintain_order=True)
    .agg(AGGREGAZIONE)
    .sort("PersonID", "anno")
)
istruzione.write_parquet(cfg.interim("istruzione_persona_anno.parquet"))
print(f"titoli: {titoli.height:,} di {titoli['PersonID'].n_unique():,} persone")
print(f"istruzione_persona_anno: {istruzione.height:,} righe x {istruzione.width} colonne")
del studi, punti, istruzione
gc.collect()


titoli: 292,576 di 173,282 persone
istruzione_persona_anno: 254,196 righe x 13 colonne


0

In [14]:
# ── 2a.5bis · controllo: l'ultima riga contro l'aggregazione di tutti i titoli ─
# L'ultima riga di istruzione_persona_anno contiene, per costruzione, tutti i
# titoli di quella persona di qualunque anno. Deve quindi coincidere esattamente,
# colonna per colonna, con l'aggregazione fatta su tutti i titoli insieme: se non
# coincide, la costruzione per anno ha perso o duplicato qualcosa.
istruzione = pl.read_parquet(cfg.interim("istruzione_persona_anno.parquet"))
ultima = (
    istruzione.sort("PersonID", "anno")
    .group_by("PersonID")
    .agg(pl.all().exclude("anno").last())
    .sort("PersonID")
)
fotografia = titoli.sort("_ordine").group_by("PersonID", maintain_order=True).agg(AGGREGAZIONE).sort("PersonID")
colonne = fotografia.columns
if not ultima.select(colonne).equals(fotografia):
    for c in colonne[1:]:
        j = fotografia.select("PersonID", c).join(ultima.select("PersonID", c), on="PersonID", suffix="_ultima")
        diverse = j.filter(pl.col(c).ne_missing(pl.col(f"{c}_ultima"))).height
        if diverse:
            print(f"ATTENZIONE {c}: {diverse:,} persone diverse")
assert ultima.select(colonne).equals(fotografia), "l'ultima riga di istruzione non coincide con l'aggregazione di tutti i titoli"
print(f"Identici: per tutte le {fotografia.height:,} persone l'ultima riga coincide con l'aggregazione di tutti i titoli")
cambia = istruzione.group_by("PersonID").agg(pl.len()).filter(pl.col("len") > 1).height
print(f"persone con l'istruzione che cambia nel tempo (piu' di un punto): {cambia:,}")
del istruzione, ultima, fotografia, titoli, AGGREGAZIONE
gc.collect()


Identici: per tutte le 173,282 persone l'ultima riga coincide con l'aggregazione di tutti i titoli
persone con l'istruzione che cambia nel tempo (piu' di un punto): 59,925


0

In [15]:
# ── 2a.6 · i titoli ricavati dal nome della persona ───────────────────────
# Se il nome contiene un titolo accademico, Highest_Degree diventa 5
# (PhD/Doctorate) scavalcando la tabella dell'istruzione, e " JD" e " MD"
# accendono Is_Law e Is_Med. ASSUNZIONE: il nome non ha una data, quindi il
# titolo vale in tutti gli anni, come un titolo senza GraduatingYear. Qui si calcolano solo i tre flag; si applicano
# dopo il join per anno, in 2b.1bis per il team e in 5.6 per il CEO.
nome = pl.col("PersonName")
ha_phd = nome.str.contains(r"Ph\.?D").fill_null(False)
# literal=True e case-sensitive di proposito, a differenza delle altre ricerche
# del blocco: in minuscolo " md" e " jd" catturerebbero pezzi di nomi comuni.
ha_jd = nome.str.contains(" JD", literal=True).fill_null(False)
ha_md = nome.str.contains(" MD", literal=True).fill_null(False)

db3 = db3.with_columns(ha_phd.alias("Nome_PhD"), ha_jd.alias("Nome_JD"), ha_md.alias("Nome_MD"))
print(f"nomi con Ph.D: {db3['Nome_PhD'].sum():,}  ' JD': {db3['Nome_JD'].sum():,}  ' MD': {db3['Nome_MD'].sum():,}")


nomi con Ph.D: 35,032  ' JD': 801  ' MD': 3,366


In [16]:
# ── 2a.7 · IsFounder ──────────────────────────────────────────────────────
posizioni = as_na(
    read_raw(cfg, "PersonPositionRelation", ["PersonID", "EntityID", "PositionLevel"]),
    R_NA_NAN,
# distinct(EntityID, PersonID): una persona puo' avere piu' posizioni nella
# stessa entita', qui ne serve una sola.
).unique(subset=["EntityID", "PersonID"], keep="first", maintain_order=True)

# Il join e' sulla COPPIA: la posizione di quella persona in QUELLA azienda.
db3 = db3.join(
    posizioni, left_on=["CompanyID", "PersonID"], right_on=["EntityID", "PersonID"], how="left"
)
del posizioni
gc.collect()

# Titolo e livello si concatenano in una stringa sola, con i mancanti resi come
# il testo "NA": cosi' la stringa non e' mai nulla e IsFounder e' sempre True o
# False, mai nullo, anche per chi non ha ne' titolo ne' posizione.
qualifica = pl.concat_str(
    [pl.col("FullTitle").fill_null("NA"), pl.col("PositionLevel").fill_null("NA")], separator="; "
)
# Si cerca "founde" oppure "founding": copre anche i refusi presenti nei dati
# (Co-Founde, Co-Founderf, Co-Foundder...) senza catturare "Foundation" e
# "Foundry", che un semplice "Found" prenderebbe.
# "Founder's Associate" e "Founders' Associate" sono assistenti del fondatore,
# non fondatori: la frase si toglie PRIMA della ricerca, cosi' chi e' insieme
# assistente e fondatore resta fondatore. Sono 3 coppie.
qualifica = qualifica.str.replace_all(r"(?i)founders?'?s?'? associate", "")
db3 = db3.with_columns(qualifica.str.contains("(?i)founde|founding").alias("IsFounder"))
# PersonName, FullTitle e PositionLevel hanno finito: servivano solo qui e in 2a.6.
db3 = db3.drop("PersonName", "FullTitle", "PositionLevel")
print(f"founder: {db3['IsFounder'].sum():,}   IsFounder nullo: {db3['IsFounder'].null_count()}")

founder: 206,117   IsFounder nullo: 0


In [17]:
# ── 2a.9 · YearFounded, e l'imputazione di StartDate ──────────────────────
# YearFounded e UltimoAnno arrivano dallo scheletro (vita_azienda, blocco 2a.1).
# Dopo il filtro di 2a.1 ogni riga ha la sua azienda, quindi nessuno dei due e' nullo.
db3 = db3.join(vita_azienda, on="CompanyID", how="left")

inizio_fondazione = pl.date(pl.col("YearFounded"), 1, 1)
inizia_prima = pl.col("StartDate").dt.year() < pl.col("YearFounded")

# ASSUNZIONE: chi non ha StartDate, o ne ha una precedente alla fondazione,
# parte dalla fondazione. Non si puo' fare altrimenti, e una presenza non puo'
# cominciare prima che l'azienda esista.
db3 = db3.with_columns(
    r_if_else(
        pl.col("StartDate").is_null() | inizia_prima,
        inizio_fondazione,
        pl.col("StartDate"),
    ).alias("StartDate")
)
print(f"StartDate valorizzate: {db3['StartDate'].is_not_null().sum():,} su {db3.height:,}")

StartDate valorizzate: 465,861 su 465,861


In [18]:
# ── 2a.10 · l'imputazione di EndDate: l'ultimo anno di vita dell'azienda ──
# Una EndDate mancante diventa il 31/12 dell'ultimo anno dello scheletro
# dell'azienda, QUALUNQUE sia IsCurrent.
# - Chi e' ancora in carica resta fino alla fine del panel dell'azienda.
# - Chi non lo e' piu' ma non ha una data di uscita resta anche lui fino alla
#   fine: meglio contare una persona in piu' che perderne una. Il costo e' un
#   Total_People piu' alto.
# - Nelle aziende fallite l'ultimo anno include la data del cambio di stato,
#   quindi la fine cade nell'anno del fallimento.
# La fine imputata non dipende dall'esito futuro dell'azienda:
# per ogni anno del panel la persona c'e' comunque, l'esito decide solo dove
# finisce il panel.
#
# Il TAGLIO. Nessuna finestra va oltre l'ultimo anno di vita: dopo MaxYear la
# fase 1 non ha OwnershipStatus ne' altre date dell'azienda, quindi quegli anni
# non potrebbero dare un valore al target.
# - I ruoli che INIZIANO dopo l'ultimo anno si scartano: cadono tutti fuori.
# - Le EndDate esplicite successive si portano all'ultimo anno.
fine_vita = pl.date(pl.col("UltimoAnno"), 12, 31)
dopo_la_fine = pl.col("StartDate").dt.year() > pl.col("UltimoAnno")
scartati = db3.select(dopo_la_fine.sum()).item()
tagliate = db3.select((pl.col("EndDate").dt.year() > pl.col("UltimoAnno")).sum()).item()
imputate = db3["EndDate"].is_null().sum()

db3 = (
    db3.filter(~dopo_la_fine)
    .with_columns(pl.min_horizontal(pl.col("EndDate").fill_null(fine_vita), fine_vita).alias("EndDate"))
    .drop("IsCurrent", "UltimoAnno")       # hanno finito il loro lavoro
)
print(f"ruoli iniziati dopo l'ultimo anno di vita, scartati: {scartati:,}")
print(f"EndDate esplicite oltre l'ultimo anno, tagliate    : {tagliate:,}")
print(f"EndDate imputate all'ultimo anno di vita           : {imputate:,}   righe: {db3.height:,}")


ruoli iniziati dopo l'ultimo anno di vita, scartati: 3,574
EndDate esplicite oltre l'ultimo anno, tagliate    : 3,502
EndDate imputate all'ultimo anno di vita           : 379,728   righe: 462,287


In [19]:
# ── 2a.11 · la finestra in anni di vita dell'azienda ──────────────────────
db3 = db3.with_columns(
    # Raddrizza le finestre impossibili: EndDate precedente a StartDate, per
    # dati sporchi all'origine o per una StartDate portata alla fondazione.
    pl.when(pl.col("EndDate") < pl.col("StartDate"))
    .then(pl.col("StartDate"))
    .otherwise(pl.col("EndDate"))
    .alias("EndDate")
).with_columns(
    # Da date di calendario a ETA' DELL'AZIENDA: e' l'unita' di misura del panel.
    (pl.col("StartDate").dt.year() - pl.col("YearFounded")).alias("DeltaStart"),
    (pl.col("EndDate").dt.year() - pl.col("YearFounded")).alias("DeltaEnd"),
)

db3 = db3.with_columns(
    # ASSUNZIONE: un founder c'e' dal primo giorno dell'azienda. La sua finestra
    # parte dall'anno di fondazione anche dove esiste una data d'inizio
    # successiva e vera, cosa che accade in 19.458 casi.
    pl.when(pl.col("IsFounder")).then(0).otherwise(pl.col("DeltaStart")).alias("DeltaStart")
).drop("StartDate", "EndDate")     # da qui in poi contano solo i due Delta

db3.write_parquet(cfg.interim("db3.parquet"))
print(f"db3: {db3.height:,} righe x {db3.width} colonne")
print(f"colonne: {db3.columns}")

db3: 462,287 righe x 10 colonne
colonne: ['CompanyID', 'PersonID', 'Gender', 'Nome_PhD', 'Nome_JD', 'Nome_MD', 'IsFounder', 'YearFounded', 'DeltaStart', 'DeltaEnd']


---
## Fase 2b — le colonne di team, anno per anno

Ogni persona viene **espansa** su tutti gli anni della sua finestra, poi si
collassa per `(azienda, anno)`.

Ne escono **13 aggregati** per anno-azienda: quante persone, la quota di donne,
gli otto flag sulle aree di studio, l'anno di laurea medio, il titolo di studio
medio, gli atenei, l'indice di esperienza medio e il numero di founder.

In [20]:
# ── 2b.1 · l'espansione a (azienda, anno, persona) ────────────────────────
db3 = pl.read_parquet(cfg.interim("db3.parquet"))

# expand_team costruisce due griglie e le unisce:
#   - per ogni AZIENDA, ogni anno da min(DeltaStart) a max(DeltaEnd) del suo team
#   - per ogni PERSONA, ogni anno della sua finestra
# left join su (azienda, anno) -> una riga per persona presente quell'anno.
# Il filtro sulla soglia sta dentro la funzione perche' e' cio' che tiene il
# risultato dentro la memoria disponibile. Ha anche un secondo effetto: su un
# anno-azienda che non ha trovato NESSUNA persona, YearFounded arriva dal lato
# persona ed e' nullo, e un confronto "> soglia" scarta i null - e' cosi' che
# non nascono righe fantasma con Total_People = 1 e tutto il resto vuoto.
# expand_team applica la soglia con un confronto STRETTO ("> soglia"), perche'
# e' condivisa con build_panel.ipynb che e' congelato e non si tocca: le si
# passa quindi l'anno precedente, che e' lo stesso insieme di aziende.
espanso = expand_team(db3, founding_year_threshold=ANNO_MIN_FONDAZIONE - 1)
del db3
gc.collect()
print(f"righe (azienda, anno, persona): {espanso.height:,}")

righe (azienda, anno, persona): 3,710,269


In [21]:
# ── 2b.1bis · esperienza e istruzione anno per anno ───────────────────────
# Ogni riga (azienda, anno, persona) prende, per quell'anno di calendario:
#   - i ruoli della persona iniziati ENTRO l'anno (tabella di 2a.3bis), da cui
#     WorkExperienceIndex: per posizioni, seggi e altri ruoli log(x+1) e
#     standardizzazione, poi la media dei tre;
#   - l'istruzione con i soli titoli conseguiti ENTRO l'anno (tabella di 2a.5),
#     poi i titoli ricavati dal nome (2a.6), che valgono in tutti gli anni.
GRUPPI = ["Posizioni", "Seggi", "AltriRuoli"]
esperienza = pl.read_parquet(cfg.interim("esperienza_persona_anno.parquet"))
istruzione = pl.read_parquet(cfg.interim("istruzione_persona_anno.parquet")).rename({"anno": "anno_istruzione"})

# L'ultima riga di ogni persona e' il suo valore alla data di estrazione: e' cosi'
# che l'interruttore spento produce la fotografia senza ricalcolare niente
# (lo verificano i controlli 2a.3ter e 2a.5bis).
def fotografia(tabella: pl.DataFrame, colonna_anno: str) -> pl.DataFrame:
    return (
        tabella.sort("PersonID", colonna_anno)
        .group_by("PersonID")
        .agg(pl.all().exclude(colonna_anno).last())
    )


espanso = (
    espanso.with_row_index("_riga")
    .with_columns((pl.col("YearFounded") + pl.col("Years")).alias("_anno"))
    # join asof: per ogni riga, la riga della stessa persona con l'anno PIU'
    # RECENTE fra quelli <= _anno. Richiede le tabelle ordinate per anno; _riga
    # serve a rimettere l'ordine originale dopo, perche' l'ordine delle righe
    # decide quello degli atenei in Institute (2b.2).
    .sort("_anno")
    .pipe(lambda d: (
        d.join_asof(esperienza.sort("anno"), left_on="_anno", right_on="anno",
                    by="PersonID", strategy="backward")
        .join_asof(istruzione.sort("anno_istruzione"), left_on="_anno", right_on="anno_istruzione",
                   by="PersonID", strategy="backward")
        if TEMPORIZZA_PERSONE else
        d.join(fotografia(esperienza, "anno"), on="PersonID", how="left")
        .join(fotografia(istruzione, "anno_istruzione"), on="PersonID", how="left")
    ))
    .sort("_riga")
    # Nessun ruolo iniziato entro quell'anno: zero ruoli.
    # Nessun titolo entro quell'anno: istruzione nulla, come per chi non ha titoli.
    .with_columns(pl.col(*GRUPPI).fill_null(0))
    # I titoli dal nome scavalcano la tabella, in ogni anno (2a.6).
    .with_columns(
        pl.when(pl.col("Nome_PhD") | pl.col("Nome_JD") | pl.col("Nome_MD")).then(5)
        .otherwise(pl.col("Highest_Degree")).alias("Highest_Degree"),
        pl.when(pl.col("Nome_JD")).then(True).otherwise(pl.col("Is_Law")).alias("Is_Law"),
        pl.when(pl.col("Nome_MD")).then(True).otherwise(pl.col("Is_Med")).alias("Is_Med"),
    )
)

# La standardizzazione, a FINESTRA ESPANSIVA. Media e deviazione standard
# campionarie si calcolano sulle coppie (persona, anno) DISTINTE - chi siede in
# due aziende nello stesso anno conta una volta sola - e solo sugli anni <= a
# quello della riga, cosi' il valore del 2005 non dipende dalle righe del 2020.
# L'anno corrente entra nella finestra: al 2005 il 2005 e' presente, non futuro.
# L'ordinamento dopo unique() non e' cosmetico: unique() restituisce le coppie in
# un ordine che cambia da un'esecuzione all'altra, e la somma cambia nelle ultime
# cifre (1e-15). Verificato: 8 ripetizioni danno 8 risultati senza ordinamento, 1 con.
log1p = {g: (pl.col(g) + 1).log() for g in GRUPPI}

if TEMPORIZZA_PERSONE:
    popolazione = espanso.unique(["PersonID", "_anno"]).sort(["PersonID", "_anno"])
    # Un passaggio per anno invece della formula con la somma dei quadrati: sono
    # ~25 passaggi su poche centinaia di migliaia di righe, e si evita la
    # cancellazione numerica che quella formula introduce.
    parametri = pl.concat([
        popolazione.filter(pl.col("_anno") <= Y).select(
            pl.lit(Y, dtype=pl.Int64).alias("_anno"),
            pl.len().alias("_n"),
            *[log1p[g].mean().alias(f"{g}_media") for g in GRUPPI],
            *[log1p[g].std(ddof=1).alias(f"{g}_dev") for g in GRUPPI],
        )
        for Y in sorted(popolazione["_anno"].unique().to_list())
    ])
    del popolazione
else:
    # A interruttore spento la popolazione e' una riga per coppia (azienda,
    # persona), non per (persona, anno), e i parametri sono unici: la fotografia
    # non ha un asse temporale su cui espandere la finestra.
    chiave = ["CompanyID", "PersonID"]
    parametri = espanso.unique(chiave).sort(chiave).select(
        *[log1p[g].mean().alias(f"{g}_media") for g in GRUPPI],
        *[log1p[g].std(ddof=1).alias(f"{g}_dev") for g in GRUPPI],
    )
parametri.write_parquet(cfg.interim("parametri_esperienza.parquet"))

COLONNE_PAR = [f"{g}_{s}" for g in GRUPPI for s in ("media", "dev")]
if TEMPORIZZA_PERSONE:
    espanso = espanso.join(parametri.drop("_n"), on="_anno", how="left")
    assert espanso.select(pl.col(COLONNE_PAR[0]).is_null().sum()).item() == 0, "anni senza parametri"
    indice = pl.mean_horizontal(
        [(log1p[g] - pl.col(f"{g}_media")) / pl.col(f"{g}_dev") for g in GRUPPI]
    )
else:
    indice = pl.mean_horizontal(
        [(log1p[g] - parametri[f"{g}_media"][0]) / parametri[f"{g}_dev"][0] for g in GRUPPI]
    )

espanso = (
    espanso.with_columns(indice.alias("WorkExperienceIndex"))
    .drop("_riga", "_anno", *GRUPPI, "Nome_PhD", "Nome_JD", "Nome_MD",
          *[c for c in ("anno", "anno_istruzione") if c in espanso.columns],
          *[c for c in COLONNE_PAR if c in espanso.columns])
)
del esperienza, istruzione
gc.collect()
if TEMPORIZZA_PERSONE:
    print(parametri.head(3))
    print(f"parametri: {parametri.height} anni, popolazione da {parametri['_n'].min():,} a {parametri['_n'].max():,} coppie")
else:
    print(parametri)
print(f"WorkExperienceIndex: media {espanso['WorkExperienceIndex'].mean():.4f} su {espanso.height:,} righe")
print(f"Highest_Degree valorizzato su {espanso['Highest_Degree'].is_not_null().mean():.1%} delle righe")


/tmp/ipykernel_400608/2445115816.py:32: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  d.join_asof(esperienza.sort("anno"), left_on="_anno", right_on="anno",


/tmp/ipykernel_400608/2445115816.py:34: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(istruzione.sort("anno_istruzione"), left_on="_anno", right_on="anno_istruzione",


shape: (3, 8)
┌───────┬───────┬──────────────┬─────────────┬─────────────┬─────────────┬───────────┬─────────────┐
│ _anno ┆ _n    ┆ Posizioni_me ┆ Seggi_media ┆ AltriRuoli_ ┆ Posizioni_d ┆ Seggi_dev ┆ AltriRuoli_ │
│ ---   ┆ ---   ┆ dia          ┆ ---         ┆ media       ┆ ev          ┆ ---       ┆ dev         │
│ i64   ┆ u32   ┆ ---          ┆ f64         ┆ ---         ┆ ---         ┆ f64       ┆ ---         │
│       ┆       ┆ f64          ┆             ┆ f64         ┆ f64         ┆           ┆ f64         │
╞═══════╪═══════╪══════════════╪═════════════╪═════════════╪═════════════╪═══════════╪═════════════╡
│ 2000  ┆ 4467  ┆ 0.584054     ┆ 0.235636    ┆ 0.032382    ┆ 0.316732    ┆ 0.380945  ┆ 0.193568    │
│ 2001  ┆ 12736 ┆ 0.588867     ┆ 0.237089    ┆ 0.033868    ┆ 0.313265    ┆ 0.380298  ┆ 0.194123    │
│ 2002  ┆ 24575 ┆ 0.591321     ┆ 0.239403    ┆ 0.035433    ┆ 0.312511    ┆ 0.381325  ┆ 0.198482    │
└───────┴───────┴──────────────┴─────────────┴─────────────┴─────────────┴───

In [22]:
# ── 2b.2 · i tredici aggregati di team ────────────────────────────────────
FLAG_AREE = ["Is_Eco", "Is_Eng", "Is_NS", "Is_Hum", "Is_SS", "Is_Med", "Is_Law", "Is_IT"]

# Gli atenei mancanti si scartano prima di concatenare, come gia' in 2a.5.
istituto = pl.col("Institute").drop_nulls()

totale = pl.len()
genere_noto = pl.col("Gender").is_not_null().sum()
team = espanso.group_by(["CompanyID", "Years"]).agg(
    # Quante persone risultano presenti in quell'anno-azienda.
    totale.alias("Total_People"),
    # Il denominatore sono le persone di GENERE NOTO: chi ha genere ignoto non
    # puo' stare al numeratore, e tenerlo al denominatore abbasserebbe la quota
    # di donne proprio nelle aziende documentate peggio. Dove nessuno ha un
    # genere noto la quota non e' definita e la colonna resta VUOTA.
    pl.when(genere_noto > 0)
    .then(pl.col("Gender").eq("Female").sum() / genere_noto * 100)
    .alias("Percent_Females"),
    # fill_null(False) prima di any(): un gruppo tutto mancante da' False, non nullo.
    *[pl.col(c).fill_null(False).any().alias(c) for c in FLAG_AREE],
    pl.col("Earliest_Year").mean().alias("Avg_Earliest_Year"),
    pl.col("Highest_Degree").mean().alias("Highest_Degree_Mean"),
    # Atenei distinti uniti con "; ". L'ordine dipende dal sort del blocco 2a.2:
    # non porta informazione - a valle si fa split(";") e poi un set - ma va
    # tenuto stabile fra un'esecuzione e l'altra.
    istituto.unique(maintain_order=True).str.join("; ").alias("Institute"),
    pl.col("WorkExperienceIndex").mean().alias("WorkExp_Idx_Mean"),
    # sum() su booleani conta i True e ignora i null.
    pl.col("IsFounder").sum().alias("Total_Founders"),
)
del espanso
gc.collect()

# Quando nessuna persona del gruppo ha un ateneo, il join di stringhe produce ""
# invece che null: si riporta a null.
team = team.with_columns(
    pl.when(pl.col("Institute") == "").then(None).otherwise(pl.col("Institute")).alias("Institute")
)
print(f"team: {team.height:,} anni-azienda x {team.width} colonne")

team: 842,010 anni-azienda x 17 colonne


In [23]:
# ── 2b.3 · il join con lo scheletro ───────────────────────────────────────
scheletro = pl.read_parquet(cfg.interim("scheletro.parquet"))
team = team.rename({"Years": "Delta"})
print(f"scheletro: {scheletro.height:,} righe   team: {team.height:,} righe")

# LEFT join: il panel E' lo scheletro, e il team si aggancia ai suoi anni. Un
# anno-azienda coperto solo dal team cadrebbe oltre MaxYear, dove non c'e'
# OwnershipStatus e quindi non c'e' un target affidabile. Dopo il taglio di
# 2a.10 nessuna finestra supera l'ultimo anno di vita, quindi il caso non si
# presenta: l'assert qui sotto lo garantisce a ogni esecuzione.
fuori = team.join(scheletro, on=["CompanyID", "Delta"], how="anti").height
assert fuori == 0, f"{fuori:,} anni-azienda del team fuori dallo scheletro"

panel = scheletro.join(team, on=["CompanyID", "Delta"], how="left").sort(["CompanyID", "Delta"])
del team, scheletro
gc.collect()

panel.write_parquet(cfg.interim("panel_team.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
print(f"righe con dati di team: {panel['Total_People'].is_not_null().sum():,}")
del panel
gc.collect()


scheletro: 907,934 righe   team: 842,010 righe


panel: 907,934 righe x 20 colonne
righe con dati di team: 842,010


0

---
## Fase 4 — deal e investitori

I round di finanziamento. Sono loro a determinare `GrowthStage`, cioè il target.

Il capitale raccolto è **`TotalRaised`**: la somma degli importi davvero
dichiarati, che tratta come zero un importo ignoto. Poiché in metà delle righe
l'importo non è dichiarato, il blocco 4.8 affianca `UndisclosedAmountShare`, che
dice quanta parte dei round di quell'anno non dichiara quanto ha raccolto: senza
di lei «non ha raccolto niente» e «non sappiamo quanto» sarebbero lo stesso zero.

Il panel non contiene nessuna stima del capitale: gli importi mancanti restano
mancanti e non vengono imputati.

In [24]:
# ── 4.1 · gli investitori, in sette categorie ─────────────────────────────
# La mappa dei tipi di investitore in sette categorie. Tutto cio' che non e' in
# mappa - compreso un tipo mancante - finisce in "Other", che non genera flag.
CATEGORIA_INVESTITORE = {
    "Venture Capital": "Venture Capital",
    "Corporate Venture Capital": "Venture Capital",
    "Growth/Expansion": "Venture Capital",
    "Not-For-Profit Venture Capital": "Venture Capital",
    "VC-Backed Company": "Venture Capital",
    "Angel (individual)": "Angel",
    "Angel Group": "Angel",
    "Accelerator/Incubator": "Accelerator",
    "Corporation": "Corporate",
    "Corporate Development": "Corporate",
    "PE/Buyout": "Private Equity",
    "Family Office": "Private Equity",
    "PE-Backed Company": "Private Equity",
    "Holding Company": "Private Equity",
    "Merchant Banking Firm": "Private Equity",
    "Mezzanine": "Private Equity",
    "Secondary Buyer": "Private Equity",
    "Other Private Equity": "Private Equity",
    "Special Purpose Acquisition Company (SPAC)": "Private Equity",
    "Fundless Sponsor": "Private Equity",
    "Government": "Public Investor",
    "University": "Public Investor",
    "Sovereign Wealth Fund": "Public Investor",
    "Mutual Fund": "Public Investor",
}
# I sei flag has_*: il nome del flag e il valore di categoria che cerca.
# Angel e Accelerator non sono fra le 53 ma servono lo stesso: al blocco 4.7
# fanno un OR con Is_Angel e Is_Accelerator, che invece lo sono.
CATEGORIE = {
    "Angel": "Angel",
    "Corporate": "Corporate",
    "VentureCapital": "Venture Capital",
    "Accelerator": "Accelerator",
    "PrivateEquity": "Private Equity",
    "PublicInvestor": "Public Investor",
}
# Due sole grandezze numeriche, ed entrambe descrivono l'INVESTITORE, non il round:
#   TotalInvestments  - quanti investimenti ha fatto in tutta la sua storia e in
#                       tutto l'universo PitchBook (mediana 2, massimo 9.927);
#   MedianRoundAmount - la dimensione mediana dei round a cui partecipa, in
#                       milioni (mediana 2,70; manca sul 18,8% degli investitori).
# Insieme dicono quanto e' grande e attivo chi mette i soldi. Diventano
# MeanTotalInvestments_cum e MeanMedianRoundAmount_cum (blocchi 4.2, 4.8 e 5.5),
# che sono due delle 53 colonne e due delle feature dei modelli.
#
# ATTENZIONE, e' un limite dichiarato: a interruttore spento sono FOTOGRAFIE alla
# data di estrazione. Nella riga del 2012 compare il numero di investimenti che
# quel fondo ha fatto fino alla data del download. La versione anno per anno
# esiste (il ramo qui sotto) ma paga un prezzo: l'estrazione contiene solo i deal
# delle aziende del campione, quindi i fondi grandi risultano molto piu' piccoli
# del vero - 20,1 investimenti dichiarati in media contro 5,5 osservati.
#
# Le altre grandezze di Investor.csv non si leggono: produrrebbero cumulate che
# nessuna colonna finale usa.
NUMERICHE = ["TotalInvestments", "MedianRoundAmount"]

relazione = as_na(
    read_raw(cfg, "DealInvestorRelation",
             ["DealID", "InvestorID", "InvestorStatus", "IsLeadInvestor"]),
    R_NA_NAN,
)
investitori = as_na(
    read_raw(cfg, "Investor", ["InvestorID", "PrimaryInvestorType", *NUMERICHE]), R_NA_NAN
).with_columns(to_num(c) for c in NUMERICHE)

relazione = relazione.join(investitori, on="InvestorID", how="left").with_columns(
    pl.col("PrimaryInvestorType")
    .replace_strict(CATEGORIA_INVESTITORE, default="Other")
    .alias("InvestorCategory")
)
del investitori

if TEMPORIZZA_INVESTITORI:
    # Le due grandezze si ricostruiscono anno per anno dai deal DATATI, e ogni
    # partecipazione prende i valori dell'investitore all'anno del suo deal.
    # Il limite: l'estrazione contiene solo i deal delle aziende del campione,
    # quindi i fondi grandi risultano piu' piccoli del vero (in media 5,1
    # investimenti ricostruiti contro 20,5 dichiarati, correlazione 0,556).
    # MedianRoundAmount si ricostruisce sulla mediana di DealSize, la colonna piu'
    # vicina alla definizione di PitchBook fra quelle disponibili.
    deal_datati = as_na(read_raw(cfg, "Deal", ["DealID", "DealDate", "DealSize"]), R_NA_NAN).with_columns(
        parse_date_r(pl.col("DealDate")).dt.year().alias("anno"), to_num("DealSize")
    ).drop_nulls("anno")
    storia = (
        as_na(read_raw(cfg, "DealInvestorRelation", ["DealID", "InvestorID"]), R_NA_NAN)
        .join(deal_datati.select("DealID", "anno", "DealSize"), on="DealID", how="inner")
    )
    # Per ogni (investitore, anno in cui ha investito) i valori CUMULATI fino a
    # quell'anno compreso: si incrociano i suoi anni con i suoi deal e si filtra.
    cumulati = (
        storia.select("InvestorID", "anno").unique()
        .join(storia.select("InvestorID", pl.col("anno").alias("_anno_deal"), "DealSize"), on="InvestorID")
        .filter(pl.col("_anno_deal") <= pl.col("anno"))
        .group_by("InvestorID", "anno")
        .agg(pl.len().cast(pl.Float64).alias("TotalInvestments"),
             pl.col("DealSize").median().alias("MedianRoundAmount"))
    )
    base = relazione.drop(*NUMERICHE).join(
        deal_datati.select("DealID", pl.col("anno").alias("_anno_deal")), on="DealID", how="left"
    )
    # join asof come in 2b.1bis: i valori dell'investitore all'anno del deal.
    # Le partecipazioni a deal senza data non hanno un anno a cui riferirsi.
    con_anno = (
        base.drop_nulls("_anno_deal").sort("_anno_deal")
        .join_asof(cumulati.sort("anno"), left_on="_anno_deal", right_on="anno",
                   by="InvestorID", strategy="backward")
        .drop("anno")
    )
    senza_anno = base.filter(pl.col("_anno_deal").is_null()).with_columns(
        *[pl.lit(None, dtype=pl.Float64).alias(c) for c in NUMERICHE]
    )
    relazione = pl.concat([con_anno, senza_anno], how="diagonal_relaxed").drop("_anno_deal")
    del deal_datati, storia, cumulati, base, con_anno, senza_anno
    gc.collect()

print(f"partecipazioni a deal: {relazione.height:,}   deal distinti: {relazione['DealID'].n_unique():,}")
print(f"investitori: {'temporizzati anno per anno' if TEMPORIZZA_INVESTITORI else 'fotografia alla data di estrazione'}")

partecipazioni a deal: 506,641   deal distinti: 278,407
investitori: fotografia alla data di estrazione


In [25]:
# ── 4.2 · aggregare per deal ──────────────────────────────────────────────
nuovo = pl.col("InvestorStatus") == "New Investor"
lead = pl.col("IsLeadInvestor") == "Yes"

# Quasi tutti gli aggregati qui sotto sono condizionati a "c'e' almeno un nuovo
# investitore in questo round?", e quella domanda ha tre esiti, non due:
#   - TRUE  se almeno uno e' "New Investor"
#   - NA    se nessuno lo e' MA qualche valore manca
#   - FALSE altrimenti.
# Usata come condizione, l'incertezza rende NULLO l'aggregato intero.
condizione = (
    pl.when(nuovo.fill_null(False).any()).then(True)
    .when(nuovo.is_null().any()).then(None)
    .otherwise(False)
)


def media_sui_nuovi(colonna: str) -> pl.Expr:
    """La media calcolata sui soli nuovi investitori, nulla se la condizione e' NA."""
    return r_if_else(condizione, pl.col(colonna).filter(nuovo.fill_null(False)).mean(), None)


def ha_categoria(categoria: str, maschera: pl.Expr) -> pl.Expr:
    """C'e' almeno un investitore di quella categoria, fra quelli selezionati
    dalla maschera? Nullo se la condizione sui nuovi investitori e' NA."""
    appartiene = pl.col("InvestorCategory").filter(maschera.fill_null(False)) == CATEGORIE[categoria]
    return r_if_else(condizione, appartiene.fill_null(False).any(), None)


per_deal = relazione.group_by("DealID").agg(
    # Quanti investitori NUOVI in questo round. Diventera' il peso delle medie
    # ponderate cumulate della fase 5.
    nuovo.fill_null(False).sum().alias("TotalInvestors"),
    media_sui_nuovi("TotalInvestments").alias("MeanTotalInvestments"),
    media_sui_nuovi("MedianRoundAmount").alias("MeanMedianRoundAmount"),
    # I sei flag sui nuovi investitori.
    *[ha_categoria(c, nuovo).alias(f"has_{c}") for c in CATEGORIE],
    # I sei flag sui lead. Attenzione all'asimmetria, che e' voluta: il FILTRO
    # seleziona i lead, ma la CONDIZIONE che decide se restituire null resta
    # quella sui nuovi investitori.
    *[ha_categoria(c, lead).alias(f"has_{c}_Lead") for c in CATEGORIE],
)
del relazione
gc.collect()
print(f"deal con almeno un investitore registrato: {per_deal.height:,}")

deal con almeno un investitore registrato: 278,407


In [26]:
# ── 4.3 · i deal ──────────────────────────────────────────────────────────
COLONNE_DEAL = [
    "CompanyID", "DealID",
    "DealNo",                  # il progressivo del round: serve alla riparazione delle date
    "DealDate",
    "DealType",                # da qui nascono i quattordici flag Is_* del blocco 4.7
    "TotalInvestedCapital",    # -> TotalRaised
    "CEOPBId",                 # -> CEO_ID, l'unico ponte fra i deal e le persone
]

deal = as_na(read_raw(cfg, "Deal", COLONNE_DEAL), R_NA_NAN).with_columns(
    pl.col("DealNo").cast(pl.Int64, strict=False),
    parse_date_r(pl.col("DealDate")).alias("DealDate"),
    to_num("TotalInvestedCapital"),
)
print(f"deal grezzi: {deal.height:,}   senza DealDate: {deal['DealDate'].is_null().sum():,}")

# Dall'anagrafica servono l'anno di fondazione (per il filtro e per collocare il
# deal) e lo stato di proprieta' con la sua data (per riparare le date mancanti).
deal = deal.join(
    pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
        "CompanyID", "YearFounded", "OwnershipStatus", "OwnershipStatusDate"
    ),
    on="CompanyID",
    how="left",
)

deal grezzi: 385,481   senza DealDate: 61,169


In [27]:
# ── 4.4 · riparare le date dei deal: quattro passaggi ─────────────────────
# 61.169 deal su 385.481 non hanno una data. Questi quattro passaggi gliene
# assegnano una stimata a 47.542; i restanti 13.627 non si agganciano a nessun
# anno e usciranno dal panel (l'elenco di chi li perde e' salvato in 4.5bis).
FALLIMENTO = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business"]
ACQUISITA = ["Acquired/Merged", "Acquired/Merged (Operating Subsidiary)"]
PRIMO_ROUND = ["Accelerator/Incubator", "Angel (individual)", "Grant", "Capitalization",
               "Early Stage VC", "Seed Round", "Spin-Off"]

data_stato = pl.col("OwnershipStatusDate")
mancanti_iniziali = deal["DealDate"].is_null().sum()

# Passaggio 1: un deal di fallimento su un'azienda che risulta fallita prende la
# data del fallimento. Plausibile: 1.210 deal.
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & pl.col("DealType").is_in(FALLIMENTO)
        & (pl.col("OwnershipStatus") == "Out of Business")
        & data_stato.is_not_null()
    ).then(data_stato).otherwise(pl.col("DealDate")).alias("DealDate")
)
# Passaggio 2: stessa idea per le acquisizioni. 234 deal.
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & (pl.col("DealType") == "Merger/Acquisition")
        & pl.col("OwnershipStatus").is_in(ACQUISITA)
        & data_stato.is_not_null()
    ).then(data_stato).otherwise(pl.col("DealDate")).alias("DealDate")
)

# Terzo e ultimo punto che filtra le aziende, con la stessa costante.
# Il filtro sta FRA il passaggio 2 e il 3: spostarlo cambierebbe quali deal
# ricevono una data stimata, quindi la posizione non si tocca.
deal = deal.filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)
print(f"deal dopo il filtro YearFounded >= {ANNO_MIN_FONDAZIONE}: {deal.height:,}")

# Passaggio 3: il PRIMO round, se e' di tipo iniziale, viene messo al 1 gennaio
# dell'anno di fondazione. E' un'assunzione forte e tocca 23.147 deal: schiaccia
# quei round sull'eta' ZERO, che e' proprio la variabile con cui a valle si
# decide chi entra nel campione (StartingAge <= 2).
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & pl.col("DealType").is_in(PRIMO_ROUND)
        & (pl.col("DealNo") == 1)
        & pl.col("YearFounded").is_not_null()
    ).then(pl.date(pl.col("YearFounded"), 1, 1)).otherwise(pl.col("DealDate")).alias("DealDate")
)

# Passaggio 4: i round senza data compresi FRA DUE ROUND DATATI vengono
# distribuiti uniformemente nell'intervallo. Due scelte:
#   - si cerca il round datato PIU' VICINO prima e dopo, non solo la riga
#     immediatamente adiacente, cosi' funziona anche quando i round senza data
#     sono due o piu' di fila: recupera 3.233 round in 1.453 aziende, 435 VC;
#   - i round dello stesso buco non prendono tutti lo stesso anno ma posizioni
#     equidistanti, che rispettano l'ordine di DealNo.
# Con un solo round nel buco la formula degenera nella media dei due limiti.
# Verificato sui round datati, fingendo che non avessero data:
#   buco da 2 round: stima esatta 48,7% contro 37,3% dando a entrambi la media;
#   buco da 3 round: 44,6% contro 31,0%.
# I limiti sono sempre round VERI: fondazione e ultimo anno dell'azienda non si
# usano, quindi i round senza nessun round datato accanto restano senza data.
deal = deal.sort(["CompanyID", "DealNo"]).with_columns(pl.col("DealDate").dt.year().alias("_anno"))
senza = pl.col("_anno").is_null()
deal = deal.with_columns(
    # il round datato piu' vicino prima e dopo, saltando quelli senza data
    pl.col("_anno").shift(1).forward_fill().over("CompanyID").alias("_prima"),
    pl.col("_anno").shift(-1).backward_fill().over("CompanyID").alias("_dopo"),
    # _buco: quanti round datati lo precedono. I round senza data fra gli stessi
    # due round datati condividono lo stesso valore, quindi lo stesso intervallo.
    (~senza).cum_sum().over("CompanyID").alias("_buco"),
)
deal = deal.with_columns(
    senza.cum_sum().over(["CompanyID", "_buco"]).alias("_posizione"),   # 1, 2, ... dentro il buco
    senza.sum().over(["CompanyID", "_buco"]).alias("_quanti"),
)
# L + (R - L) * i / (k + 1), arrotondato al piu' vicino: con k = 1 e' la media
# dei due limiti, arrotondata per eccesso.
passo = (pl.col("_dopo") - pl.col("_prima")) * pl.col("_posizione") / (pl.col("_quanti") + 1)
stima = (pl.col("_prima") + passo + 0.5).floor().cast(pl.Int64)
deal = deal.with_columns(
    pl.when(senza & pl.col("_prima").is_not_null() & pl.col("_dopo").is_not_null())
    .then(pl.date(stima, 1, 1))
    .otherwise(pl.col("DealDate"))
    .alias("DealDate")
).drop("_anno", "_prima", "_dopo", "_buco", "_posizione", "_quanti",
       "OwnershipStatus", "OwnershipStatusDate")

print(f"date mancanti: {mancanti_iniziali:,} -> {deal['DealDate'].is_null().sum():,}")

deal dopo il filtro YearFounded >= 2000: 337,898


date mancanti: 61,169 -> 13,627


In [28]:
# ── 4.5 · collocare il deal in un anno del panel ──────────────────────────
anno_deal = pl.col("DealDate").dt.year()

deal = deal.with_columns(
    # Se l'anno del deal manca il risultato resta NULLO: il deal finisce in un
    # gruppo ad anno nullo e al join
    # con il panel non si aggancia a niente. Spariscono cosi', in silenzio,
    # 13.627 deal su 12.042 aziende, di cui 3.380 VC, che avrebbero alzato lo
    # stadio di quelle aziende. Il blocco 4.5bis li registra uno per uno invece
    # di lasciarli sparire senza traccia.
    # pl.max_horizontal da solo ignora i null e restituirebbe l'anno di
    # fondazione, parcheggiando quei deal sull'anno zero: comportamento diverso,
    # e nemmeno quello giusto. Il when esplicito serve a non farlo.
    pl.when(anno_deal.is_null())
    .then(None)
    .otherwise(pl.max_horizontal(anno_deal, pl.col("YearFounded")))
    .alias("Year_Delta"),
)
# I deal datati PRIMA della fondazione vengono schiacciati sull'ANNO DI
# FONDAZIONE - non sull'anno zero, che e' il comportamento alternativo descritto
# qui sopra - perche' il panel comincia a Age = 0 e prima non esiste una riga su
# cui atterrare. Scartarli costerebbe di piu' che spostarli.
# *Misurato: 139 deal su 284.345 datati (lo 0,049%), 121 aziende,
# di cui 30 non hanno nessun altro deal datato. Precedono la fondazione di 1
# anno nel 56,8% dei casi e di 2 o meno nel 76,3%. Il 68% sono acceleratori,
# grant, seed, angel e crowdfunding: eventi VERI, che precedono la costituzione
# legale. Solo 17 sono impossibili (9 Merger/Acquisition e 8 Out of Business).*
print(f"deal senza anno, che escono dal panel: {deal['Year_Delta'].is_null().sum():,}")

deal = deal.join(per_deal, on="DealID", how="left")
del per_deal
gc.collect()
# I token che valgono "mancante", riapplicati dopo il join.
deal = as_na(deal, R_NA_NAN)

deal senza anno, che escono dal panel: 13,627


In [29]:
# ── 4.5bis · l'elenco delle aziende con round senza data ──────────────────
# I round rimasti senza anno non si agganciano a nessuna riga del panel (4.9) e
# spariscono senza errore e senza traccia. Qui si salva CHI li perde e quanti,
# in un file a parte: il panel non cambia di una riga ne' di una colonna.
# Serve al controllo di robustezza dell'articolo ("rifacendo le stime senza
# queste aziende i risultati tengono") e a quantificare il bias sul target, che
# va in una sola direzione: un round non datato non puo' far salire di stadio.
VC = ["Early Stage VC", "Later Stage VC", "Seed Round"]
senza_anno = pl.col("Year_Delta").is_null()
aziende_round_persi = (
    deal.group_by("CompanyID")
    .agg(
        pl.len().alias("round_totali"),
        senza_anno.sum().alias("round_senza_data"),
        (senza_anno & pl.col("DealType").is_in(VC)).sum().alias("round_vc_senza_data"),
    )
    .filter(pl.col("round_senza_data") > 0)
    # perde_tutti: nel panel questa azienda risulta senza NESSUN finanziamento,
    # e senza stadio non entra nemmeno nel dataset dei modelli.
    .with_columns((pl.col("round_senza_data") == pl.col("round_totali")).alias("perde_tutti"))
    .sort("CompanyID")
)
aziende_round_persi.write_parquet(cfg.interim("aziende_round_senza_data.parquet"))
print(f"aziende con round senza data: {aziende_round_persi.height:,}"
      f"   di cui perdono tutti i round: {aziende_round_persi['perde_tutti'].sum():,}")
print(f"round persi: {aziende_round_persi['round_senza_data'].sum():,}"
      f"   di cui VC: {aziende_round_persi['round_vc_senza_data'].sum():,}")
del aziende_round_persi


aziende con round senza data: 12,042   di cui perdono tutti i round: 757
round persi: 13,627   di cui VC: 3,380


In [30]:
# ── 4.7 · i quattordici flag sul tipo di deal ─────────────────────────────
# Otto NON sono fra le 53 ma sono la cascata che produce GrowthStage, cioe' il
# target: Is_Preseed, Is_Seed, Is_EarlyVC, Is_LaterVC, Is_MA, Is_Public_Exit,
# Is_Out, Is_PE. Sei lo sono: Is_Debt, Is_Grant, Is_SpinOff, Is_CrowdFunding,
# Is_Accelerator, Is_Angel.
PRESEED = ["Accelerator/Incubator", "Angel (individual)", "Grant", "Spin-Off",
           "Equity Crowdfunding", "Product Crowdfunding", "Capitalization"]
USCITA_MA = ["Merger/Acquisition", "Buyout/LBO", "Debt - Acquisition", "Debt - Merger",
             "Merger of Equals", "Investor Buyout by Management", "Corporate Asset Purchase",
             "Reverse Merger"]
USCITA_PUBBLICA = ["IPO", "Secondary Transaction - Open Market",
                   "Secondary Transaction - Stock Distribution",
                   "Public Investment 2nd Offering", "PIPE"]
FALLIMENTI = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business",
              "Restart - Angel", "Restart - Early VC", "Restart - Later VC"]
DEBITO = ["Debt - General", "Debt Conversion", "Mezzanine", "Convertible Debt",
          "Debt Refinancing", "Debt - PPP", "Debt Repayment", "Dividend Recapitalization",
          "Exit Financing", "Project Financing", "Share Repurchase", "Leveraged Recapitalization"]
PRIVATE_EQUITY = ["PE Growth/Expansion", "Secondary Transaction - Private", "Corporate",
                  "Platform Creation", "GP Stakes", "General Corporate Purpose", "Capital Spending"]

FLAG_DEAL = {
    "Is_Preseed": PRESEED,
    "Is_Seed": ["Seed Round"],
    "Is_EarlyVC": ["Early Stage VC"],
    "Is_LaterVC": ["Later Stage VC"],
    "Is_MA": USCITA_MA,
    "Is_Public_Exit": USCITA_PUBBLICA,
    "Is_Out": FALLIMENTI,
    "Is_Debt": DEBITO,
    "Is_PE": PRIVATE_EQUITY,
    "Is_Grant": ["Grant"],
    "Is_SpinOff": ["Spin-Off"],
    "Is_CrowdFunding": ["Equity Crowdfunding", "Product Crowdfunding"],
    "Is_Accelerator": ["Accelerator/Incubator"],
    "Is_Angel": ["Angel (individual)"],
}
# is_in su un DealType nullo darebbe null: fill_null(False) lo rende falso.
deal = deal.with_columns(
    *[pl.col("DealType").is_in(v).fill_null(False).alias(f) for f, v in FLAG_DEAL.items()]
)

In [31]:
# ── 4.8 · aggregare a (azienda, anno) ─────────────────────────────────────
importo = pl.col("TotalInvestedCapital")


def qualunque(colonna: str) -> pl.Expr:
    """Almeno un True nel gruppo. Un gruppo tutto mancante da' False, non nullo."""
    return pl.col(colonna).fill_null(False).any()


deals_panel = deal.group_by(["CompanyID", "Year_Delta"]).agg(
    pl.len().alias("N_Deal"),
    # Un importo ignoto vale ZERO: e' una semantica che confonde "non ha
    # raccolto niente" con "non sappiamo quanto". La colonna qui sotto serve
    # proprio a rendere distinguibili i due casi.
    importo.fill_null(0.0).sum().alias("TotalRaised"),
    # UndisclosedAmountShare: la quota di round dell'anno con l'importo NON
    # dichiarato, cioe' quanto e' affidabile il TotalRaised di quella riga. In
    # meta' delle righe del dataset l'importo non e' dichiarato. Non e' mai
    # nulla, quindi non incontra la soglia del 40% di mancanti che a valle
    # scarterebbe la colonna.
    importo.is_null().mean().alias("UndisclosedAmountShare"),
    # I quattordici flag: acceso se ALMENO UN deal di quell'anno lo accende.
    *[qualunque(f).alias(f) for f in FLAG_DEAL if f not in ("Is_Accelerator", "Is_Angel")],
    # Queste due sono un OR fra il tipo di deal e la categoria dell'investitore:
    # un round da un acceleratore conta anche se il DealType non lo dice.
    (qualunque("Is_Accelerator") | qualunque("has_Accelerator")).alias("Is_Accelerator"),
    (qualunque("Is_Angel") | qualunque("has_Angel")).alias("Is_Angel"),
    # Quanti investitori nuovi in totale nell'anno: e' il peso delle medie
    # ponderate cumulate della fase 5.
    pl.col("TotalInvestors").fill_null(0).sum().alias("TotalInvestors"),
    pl.col("MeanTotalInvestments").mean().alias("MeanTotalInvestments"),
    pl.col("MeanMedianRoundAmount").mean().alias("MeanMedianRoundAmount"),
    # I quattro has_* sui nuovi investitori e i sei sui lead, tutti nelle 53.
    *[qualunque(f"has_{c}").alias(f"has_{c}")
      for c in ("Corporate", "VentureCapital", "PrivateEquity", "PublicInvestor")],
    *[qualunque(f"has_{c}_Lead").alias(f"has_{c}_Lead") for c in CATEGORIE],
    # L'ultimo CEO non nullo, nell'ordine delle righe.
    tail_na_omit("CEOPBId").alias("CEO_ID"),
)
del deal
gc.collect()
print(f"deals_panel: {deals_panel.height:,} anni-azienda con almeno un deal")
print(f"  nel gruppo ad anno nullo (non si agganceranno): "
      f"{deals_panel.filter(pl.col('Year_Delta').is_null()).height:,}")

deals_panel: 286,517 anni-azienda con almeno un deal
  nel gruppo ad anno nullo (non si agganceranno): 12,042


In [32]:
# ── 4.9 · innestare nel panel, e TR_D ─────────────────────────────────────
panel = pl.read_parquet(cfg.interim("panel_team.parquet"))

panel = panel.join(deals_panel, on=["CompanyID", "Year_Delta"], how="left").with_columns(
    # TR_D = 1 quando quell'anno-azienda non ha avuto NESSUN deal. Basta
    # guardare TotalRaised: dentro deals_panel e' una somma che tratta i
    # mancanti come zero, quindi non e' mai nulla, e diventa nulla solo qui,
    # quando il join non trova niente da agganciare.
    pl.col("TotalRaised").is_null().cast(pl.Int64).alias("TR_D"),
)
del deals_panel
gc.collect()

# Qui i token da convertire includono anche "Inf" e "-Inf": sono i -Inf prodotti
# da max() e i NaN prodotti da mean() sui gruppi tutti mancanti.
panel = as_na(panel, R_NA_INF)
panel.write_parquet(cfg.interim("panel_deals.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
print(f"TR_D = 1 (anno senza deal): {(panel['TR_D'] == 1).sum():,}")
del panel
gc.collect()

panel: 907,934 righe x 52 colonne
TR_D = 1 (anno senza deal): 635,259


0

---
## Fase 5 — finalizzazione

I flag diventano **cumulativi** (una volta acceso, resta acceso), nasce
`GrowthStage`, e si agganciano i tre attributi del CEO.

Le cumulate ponderate dicono quanto erano grandi e attivi, in media, gli
investitori entrati fino a quell'anno; l'anagrafica dell'azienda (paese e
settore) si aggancia qui, perché vive a livello azienda e non per anno.

In [33]:
# ── 5.1 · i ventiquattro flag diventano cumulativi ────────────────────────
# cumany: un flag acceso in un anno resta acceso in tutti gli anni successivi.
# E' cosi' che "l'azienda ha gia' fatto un round seed" diventa uno stato e non
# un evento. Rende gli stadi monotoni: un'azienda non puo' retrocedere.
FLAG_CUMULATIVI = [
    "Is_Preseed", "Is_Seed", "Is_EarlyVC", "Is_LaterVC", "Is_MA", "Is_Public_Exit",
    "Is_Out", "Is_Debt", "Is_PE", "Is_Grant", "Is_SpinOff", "Is_CrowdFunding",
    "Is_Accelerator", "Is_Angel",
    "has_Corporate", "has_VentureCapital", "has_PrivateEquity", "has_PublicInvestor",
    "has_Angel_Lead", "has_Corporate_Lead", "has_VentureCapital_Lead",
    "has_Accelerator_Lead", "has_PrivateEquity_Lead", "has_PublicInvestor_Lead",
]
# has_Angel e has_Accelerator non sono in questa lista: sono gia' stati
# consumati dall'OR del blocco 4.8 e non servono piu'.

panel = pl.read_parquet(cfg.interim("panel_deals.parquet")).sort(["CompanyID", "Year_Delta"])
# .over("CompanyID"): la cumulata riparte da capo per ogni azienda.
panel = panel.with_columns(cumany(pl.col(c)).over("CompanyID").alias(c) for c in FLAG_CUMULATIVI)
print(f"flag resi cumulativi: {len(FLAG_CUMULATIVI)}")

flag resi cumulativi: 24


In [34]:
# ── 5.2 · GrowthStage, la cascata che definisce il target ─────────────────
stato = pl.col("OwnershipStatus")
out, pubblica, ma = pl.col("Is_Out"), pl.col("Is_Public_Exit"), pl.col("Is_MA")
later, pe = pl.col("Is_LaterVC"), pl.col("Is_PE")
early, seed, preseed = pl.col("Is_EarlyVC"), pl.col("Is_Seed"), pl.col("Is_Preseed")
non_terminale = ~ma & ~pubblica & ~out

# Sette condizioni a CORTO CIRCUITO: il primo match vince, l'ordine e' vincolante.
# ASSUNZIONE: qui si mescolano due fonti di natura diversa.
#   - OwnershipStatus e' lo stato ATTUALE dell'azienda, agganciato al solo anno
#     della sua data (2.764 righe in tutto);
#   - i flag sono storici e cumulativi.
# Nei primi tre rami la prima fonte vince con un OR.
# Attenzione alla logica a tre valori: quando OwnershipStatus e' nullo,
# "null | True" vale True e il flag decide, mentre "null | False" vale null e la
# condizione non matcha, e il ramo successivo viene provato.
panel = panel.with_columns(
    pl.when((stato == "Out of Business") | out).then(pl.lit("Out"))
    .when(stato.is_in(["Publicly Held", "In IPO Registration"]) | pubblica).then(pl.lit("Exit_Public"))
    .when(stato.is_in(["Acquired/Merged", "Acquired/Merged (Operating Subsidiary)"]) | ma).then(pl.lit("Exit_M&A"))
    .when((later | pe) & non_terminale).then(pl.lit("LaterVC_or_Other"))
    .when(early & ~later & non_terminale & ~pe).then(pl.lit("EarlyVC"))
    .when(seed & ~early & ~later & non_terminale & ~pe).then(pl.lit("Seed"))
    .when(preseed & ~seed & ~early & ~later & non_terminale & ~pe).then(pl.lit("Preseed"))
    .otherwise(None)
    .alias("GrowthStage")
)
print(panel["GrowthStage"].value_counts(sort=True))

shape: (8, 2)
┌──────────────────┬────────┐
│ GrowthStage      ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ null             ┆ 240900 │
│ Preseed          ┆ 217654 │
│ EarlyVC          ┆ 154334 │
│ LaterVC_or_Other ┆ 112010 │
│ Seed             ┆ 78386  │
│ Exit_M&A         ┆ 57028  │
│ Out              ┆ 35222  │
│ Exit_Public      ┆ 12400  │
└──────────────────┴────────┘


In [35]:
# ── 5.3 · dove non c'e' nessun deal, il raccolto e' zero ──────────────────
# Un anno senza nessun round non ha raccolto niente, e non ha nemmeno importi
# non dichiarati: l'indicatore vale zero, non nullo.
panel = panel.with_columns(
    pl.when(pl.col("TR_D") == 1).then(0.0).otherwise(pl.col("TotalRaised")).alias("TotalRaised"),
    pl.when(pl.col("TR_D") == 1).then(0.0)
    .otherwise(pl.col("UndisclosedAmountShare")).alias("UndisclosedAmountShare"),
)

# ── 5.4 · le cumulate ─────────────────────────────────────────────────────
# TotalInvestors arriva dalla fase 4 come "nuovi investitori di QUESTO anno".
# Va letto PRIMA di essere sovrascritto dalla propria cumulata, perche' e' il
# peso delle medie ponderate del blocco 5.5. Servono due with_columns separati,
# perche' polars valuta tutte le espressioni di un singolo with_columns sul
# frame di partenza: nello stesso blocco leggerebbe il valore gia' sovrascritto.
panel = panel.with_columns(pl.col("TotalInvestors").fill_null(0).alias("NewInvestors"))

panel = panel.with_columns(
    # r_cum_sum propaga un nullo a tutto il resto del gruppo: cumsum(1, null, 3)
    # da' (1, null, null). Qui i fill_null(0) tolgono ogni nullo, quindi
    # coinciderebbe con cum_sum, ma si usa la primitiva che rende esplicita la
    # semantica voluta se un giorno il fill sparisse.
    r_cum_sum(pl.col("N_Deal").fill_null(0)).over("CompanyID").alias("N_Deal"),
    r_cum_sum(pl.col("NewInvestors").fill_null(0)).over("CompanyID").alias("TotalInvestors"),
)
print(f"N_Deal massimo: {panel['N_Deal'].max()}   TotalInvestors massimo: {panel['TotalInvestors'].max()}")

N_Deal massimo: 32   TotalInvestors massimo: 191


In [36]:
# ── 5.5 · le medie ponderate cumulate ─────────────────────────────────────
# "In media, quanto erano grandi gli investitori che hanno messo soldi in questa
# azienda fino a quest'anno", pesato per quanti erano.
# ATTENZIONE: a interruttore spento i due attributi di partenza sono fotografie
# alla data di estrazione, quindi la cumulata e' temporizzata ma il suo contenuto
# no (vedi il commento in 4.1).
# weighted_cumulative usa l'identita' algebrica cumsum(x*w) / cumsum(w) sulle
# sole righe valide: O(n) ed esatta, non un'approssimazione.
PONDERATE = ["MeanTotalInvestments", "MeanMedianRoundAmount"]

panel = panel.with_columns(
    weighted_cumulative(c, "NewInvestors", ["CompanyID"]).alias(f"{c}_cum") for c in PONDERATE
)
# Le versioni non cumulate hanno finito: nelle 53 ci sono solo le _cum.
panel = panel.drop(*PONDERATE, "NewInvestors")

In [37]:
# ── 5.6 · gli attributi del CEO ───────────────────────────────────────────
# Del CEO arrivano nel panel tre attributi: genere, indice di esperienza e
# titolo di studio piu' alto. Quest'ultimo e' nullo su circa due terzi delle
# righe e a valle puo' essere scartato dalla soglia del 40% di mancanti; si
# tiene comunque, perche' la selezione delle feature avviene fuori di qui.
CEO_VARS = ["Gender"]

# CEO_ID e' dichiarato solo negli anni in cui c'e' stato un deal. Si propaga in
# avanti: chi era CEO all'ultimo round lo resta finche' non ne arriva un altro.
# Prima di propagarlo si tengono da parte due cose che servono sotto: CHI e
# QUANDO il primo deal dichiara come CEO, e l'anno dell'ultima dichiarazione.
panel = panel.sort("CompanyID", "Year_Delta").with_columns(
    pl.when(pl.col("CEO_ID").is_not_null()).then(pl.col("Year_Delta")).otherwise(None)
      .fill_null(strategy="forward").over("CompanyID").alias("_anno_dich"),
)
_primo = (
    panel.filter(pl.col("CEO_ID").is_not_null())
    .group_by("CompanyID")
    .agg(pl.col("Year_Delta").min().alias("_primo_anno"),
         pl.col("CEO_ID").sort_by("Year_Delta").first().alias("_primo_ceo"))
)
panel = (
    panel.join(_primo, on="CompanyID", how="left")
    .with_columns(pl.col("CEO_ID").fill_null(strategy="forward").over("CompanyID"))
)

# ── Il buco: prima del primo round non esiste nessun CEO ──────────────────
# Il CEO viene dai deal, quindi negli anni precedenti al primo round non c'e'.
# Sono proprio gli anni iniziali, dove i modelli leggono le feature: *misurato,
# la copertura scende dal 70,8% del panel al 52,4% fra le righe con Age <= 2.*
#
# Si tappa con i ruoli da CEO del board team (tabella di 2a.1), ma SOLO dove
# l'attribuzione e' verificabile, perche' il titolo del board e' una fotografia
# alla data di estrazione, e proiettarlo all'indietro senza controlli
# reintrodurrebbe il look-ahead che la temporizzazione serve a togliere.
# Le condizioni, tutte necessarie:
#   1. un SOLO ruolo da CEO copre quell'anno (niente co-CEO da arbitrare);
#   2. il titolo e' founder + CEO. *Per un founder la StartDate coincide con la
#      fondazione nell'89,0% dei casi, contro il 22,7% dei CEO non founder:
#      quindi per lui "da quando e' in azienda" non e' ambiguo.*
#   3. ha una StartDate VERA, non imputata;
#   4. e' la STESSA persona che il primo deal conferma CEO: il deal fa da
#      conferma indipendente, e delimita chi, mentre la data delimita da quando;
#   5. nessun'altra persona ha un ruolo da CEO cominciato PRIMA di lui.
#      *Sono 246 aziende su 51.470, lo 0,48%: sono i controesempi, dove il dato
#      stesso smentisce che il founder sia sempre stato CEO. Si escludono invece
#      di assumerli via.*
# *Misurato: riempie 71.593 righe e porta la copertura a Age <= 2 dal 52,4% al
# 66,9%. Il riempimento indiscriminato arriverebbe al 74,1%, ma con il 72% delle
# righe fondate su un "CEO adesso" proiettato indietro di 8 anni mediani.*
ruoli_ceo = pl.read_parquet(cfg.interim("ruoli_ceo.parquet"))
_attivi = (
    panel.select("CompanyID", "Year_Delta")
    .join_where(
        ruoli_ceo,
        pl.col("CompanyID") == pl.col("CompanyID_right"),
        pl.col("Year_Delta") >= pl.col("da"),
        pl.col("Year_Delta") <= pl.col("a"),
    )
    .group_by("CompanyID", "Year_Delta")
    .agg(pl.col("PersonID").n_unique().alias("_n"),
         pl.col("PersonID").first().alias("_board"),
         pl.col("sv").first().alias("_sv"),
         pl.col("is_founder").first().alias("_f"))
    .filter(pl.col("_n") == 1)
)
# Condizione 5: aziende in cui un ALTRO ha un ruolo da CEO iniziato prima.
_prima = (
    ruoli_ceo.filter(pl.col("sv").is_not_null())
    .join(ruoli_ceo.filter(pl.col("sv").is_not_null()), on="CompanyID")
    .filter((pl.col("PersonID") != pl.col("PersonID_right")) & (pl.col("sv_right") < pl.col("sv")))
    .select("CompanyID", pl.col("PersonID").alias("_board")).unique()
    .with_columns(pl.lit(True).alias("_scartata"))
)
panel = (
    panel.join(_attivi, on=["CompanyID", "Year_Delta"], how="left")
    .join(_prima, on=["CompanyID", "_board"], how="left")
)

_riempibile = (
    pl.col("CEO_ID").is_null()
    & pl.col("_primo_anno").is_not_null() & (pl.col("Year_Delta") < pl.col("_primo_anno"))
    & (pl.col("_n") == 1) & pl.col("_f") & pl.col("_sv").is_not_null()
    & (pl.col("_board") == pl.col("_primo_ceo"))
    & pl.col("_scartata").is_null()
)
# ── I cambi di CEO che i deal non vedono ──────────────────────────────────
# Quando un nuovo CEO entra fra un round e l'altro, il riporto in avanti
# continua a trascinare il vecchio fino al round successivo: in quelle righe
# oggi attribuiamo la persona SBAGLIATA. Se un ruolo del board comincia con
# data vera dopo l'ultima dichiarazione dei deal ed e' un'altra persona, vince
# lui. *Misurato: 3.497 righe in 1.298 aziende, il vecchio CEO veniva trascinato
# oltre l'inizio del nuovo per 1 anno mediano.*
_da_correggere = (
    pl.col("CEO_ID").is_not_null() & (pl.col("_n") == 1) & pl.col("_sv").is_not_null()
    & (pl.col("_sv") > pl.col("_anno_dich")) & (pl.col("_board") != pl.col("CEO_ID"))
)
_riempite = panel.select(_riempibile.sum()).item()
_corrette = panel.select(_da_correggere.sum()).item()
panel = panel.with_columns(
    pl.when(_riempibile).then(pl.col("_board"))
    .when(_da_correggere).then(pl.col("_board"))
    .otherwise(pl.col("CEO_ID")).alias("CEO_ID")
).drop("_n", "_board", "_sv", "_f", "_scartata", "_primo_anno", "_primo_ceo", "_anno_dich")
del ruoli_ceo, _attivi, _prima, _primo
gc.collect()
print(f"CEO riempiti prima del primo round: {_riempite:,}   attribuzioni corrette: {_corrette:,}")

ceo = (
    pl.read_parquet(cfg.interim("db3.parquet"))
    .select("CompanyID", "PersonID", *CEO_VARS, "Nome_PhD", "Nome_JD", "Nome_MD")
    .rename({c: f"{c}_CEO" for c in CEO_VARS})
    .with_columns(pl.lit(True).alias("_ceo_nel_team"),
                  # Titolo ricavato dal nome del CEO (2a.6): vale in tutti gli anni.
                  (pl.col("Nome_PhD") | pl.col("Nome_JD") | pl.col("Nome_MD")).alias("_ceo_dottore"))
    .drop("Nome_PhD", "Nome_JD", "Nome_MD")
)
# Il join e' sulla COPPIA (azienda, persona): il CEO deve essere nel board team
# di quella stessa azienda. Misurato: lo e' nel 100% dei casi dichiarati.
panel = panel.join(
    ceo, left_on=["CompanyID", "CEO_ID"], right_on=["CompanyID", "PersonID"], how="left"
)

# WorkExperienceIndex_CEO e Highest_Degree_CEO nell'anno della riga, come in
# 2b.1bis: i ruoli del CEO iniziati entro quell'anno (con la stessa formula e gli
# stessi parametri) e il titolo piu' alto conseguito entro quell'anno. Restano
# nulli dove il CEO non e' nel team di quell'azienda, come gli altri attributi.
GRUPPI = ["Posizioni", "Seggi", "AltriRuoli"]
parametri = pl.read_parquet(cfg.interim("parametri_esperienza.parquet"))
esperienza_ceo = pl.read_parquet(cfg.interim("esperienza_persona_anno.parquet"))
istruzione_ceo = (
    pl.read_parquet(cfg.interim("istruzione_persona_anno.parquet"),
                    columns=["PersonID", "anno", "Highest_Degree"])
    .rename({"anno": "anno_istruzione"})
)
# Stessi due rami di 2b.1bis: a interruttore spento vale l'ultima riga della
# persona, cioe' la fotografia alla data di estrazione.
ultima_riga = lambda t, a: t.sort("PersonID", a).group_by("PersonID").agg(pl.all().exclude(a).last())
panel = (
    panel.with_row_index("_riga")
    .sort("Year_Delta")
    .pipe(lambda d: (
        d.join_asof(esperienza_ceo.sort("anno"), left_on="Year_Delta", right_on="anno",
                    by_left="CEO_ID", by_right="PersonID", strategy="backward")
        .join_asof(istruzione_ceo.sort("anno_istruzione"), left_on="Year_Delta",
                   right_on="anno_istruzione", by_left="CEO_ID", by_right="PersonID", strategy="backward")
        if TEMPORIZZA_PERSONE else
        d.join(ultima_riga(esperienza_ceo, "anno"), left_on="CEO_ID", right_on="PersonID", how="left")
        .join(ultima_riga(istruzione_ceo, "anno_istruzione"), left_on="CEO_ID", right_on="PersonID", how="left")
    ))
    .sort("_riga")
    .with_columns(pl.col(*GRUPPI).fill_null(0))
)

# Gli STESSI parametri di 2b.1bis. A interruttore acceso sono per anno (finestra
# espansiva), quindi si agganciano sull'anno della riga; i pochi anni del panel
# che non compaiono nell'espansione del team prendono i parametri dell'anno
# disponibile piu' vicino, altrimenti l'indice del CEO resterebbe nullo li'.
COLONNE_PAR = [f"{g}_{s}" for g in GRUPPI for s in ("media", "dev")]
if TEMPORIZZA_PERSONE:
    panel = (
        panel.sort("Year_Delta")
        .join(parametri.drop("_n"), left_on="Year_Delta", right_on="_anno", how="left")
        .with_columns(pl.col(COLONNE_PAR).fill_null(strategy="forward"))
        .with_columns(pl.col(COLONNE_PAR).fill_null(strategy="backward"))
        .sort("_riga")
    )
    assert panel.select(pl.col(COLONNE_PAR[0]).is_null().sum()).item() == 0, "anni senza parametri"
    indice_ceo = pl.mean_horizontal(
        [((pl.col(g) + 1).log() - pl.col(f"{g}_media")) / pl.col(f"{g}_dev") for g in GRUPPI]
    )
else:
    indice_ceo = pl.mean_horizontal(
        [((pl.col(g) + 1).log() - parametri[f"{g}_media"][0]) / parametri[f"{g}_dev"][0] for g in GRUPPI]
    )

panel = panel.with_columns(
    pl.when(pl.col("_ceo_nel_team")).then(indice_ceo).alias("WorkExperienceIndex_CEO"),
    pl.when(pl.col("_ceo_nel_team"))
    .then(pl.when(pl.col("_ceo_dottore")).then(5).otherwise(pl.col("Highest_Degree")))
    .alias("Highest_Degree_CEO"),
)
panel = panel.drop("_riga", *GRUPPI, "Highest_Degree", "_ceo_nel_team", "_ceo_dottore", "CEO_ID",
                   *[c for c in ("anno", "anno_istruzione") if c in panel.columns],
                   *[c for c in COLONNE_PAR if c in panel.columns])
del ceo, parametri, esperienza_ceo, istruzione_ceo
gc.collect()
print(f"righe con attributi del CEO: {panel['Gender_CEO'].is_not_null().sum():,}")

CEO riempiti prima del primo round: 71,600   attribuzioni corrette: 3,491


/tmp/ipykernel_400608/3346575851.py:143: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  d.join_asof(esperienza_ceo.sort("anno"), left_on="Year_Delta", right_on="anno",


/tmp/ipykernel_400608/3346575851.py:145: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(istruzione_ceo.sort("anno_istruzione"), left_on="Year_Delta",


righe con attributi del CEO: 710,455


In [38]:
# ── 5.7 · le due colonne d'anagrafica, e la selezione ─────────────────────
# HQCountry e PrimaryIndustrySector sono due delle 53 e non sono mai entrate nel
# panel: vivono a livello azienda, e qui si agganciano.
panel = panel.join(
    pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
        "CompanyID", "HQCountry", "PrimaryIndustrySector"
    ),
    on="CompanyID",
    how="left",
)
# db_master_1 ha una riga per azienda, quindi il join non puo' moltiplicare le
# righe. Se lo facesse sarebbe un errore grave e silenzioso: meglio accorgersene.
righe_attese = panel.height

# Age e' l'eta' dell'azienda, cioe' Delta con un altro nome. E' una delle 53;
# Delta e Year_Delta restano come coordinate interne fino alla fine della fase 7.
panel = panel.with_columns(pl.col("Delta").alias("Age")).drop(
    # OwnershipStatus ha fatto il suo lavoro nella cascata GrowthStage.
    "OwnershipStatus", "Delta", "TR_D",
)
assert panel.height == righe_attese, "il join con l'anagrafica ha moltiplicato le righe"
panel.write_parquet(cfg.interim("panel_finale.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
del panel
gc.collect()

panel: 907,934 righe x 55 colonne


0

---
## Fase 6 — raggruppamento degli stadi e troncamento

I sette valori di `GrowthStage` si raggruppano in quattro (`Early`, `Later`,
`Exit`, `Out`); si calcola lo stadio **futuro** di ogni azienda — prima del
troncamento, perché dopo quell'informazione non esisterebbe più — e infine il
panel si taglia all'uscita: dalla prima riga terminale in poi le righe
spariscono, compresa quella dell'uscita stessa.

**Un'assunzione da conoscere.** Le regole di questa fase non sono state
trascritte da un sorgente: sono state ricostruite a partire da un output già
calcolato e verificate contro di esso riga per riga, a divergenza zero. Reggono
su tutti i casi osservati, ma nessuno può garantire che una combinazione mai
comparsa in quel file venga trattata come ci si aspetta.

In [39]:
# ── 6.1 · dai sette stadi ai quattro gruppi ───────────────────────────────
GRUPPO_STADIO = {
    "Preseed": "Early",
    "Seed": "Early",
    "EarlyVC": "Early",
    "LaterVC_or_Other": "Later",
    "Out": "Out",
    "Exit_M&A": "Exit",
    "Exit_Public": "Exit",
}
GRUPPI_TERMINALI = ["Out", "Exit"]

panel = pl.read_parquet(cfg.interim("panel_finale.parquet")).sort(["CompanyID", "Year_Delta"])
# default=None: uno stadio nullo resta nullo, non diventa una categoria.
panel = panel.with_columns(
    pl.col("GrowthStage").replace_strict(GRUPPO_STADIO, default=None).alias("GrowthStageGroup")
).drop("GrowthStage")   # la versione a sette stadi non e' fra le 53
print(panel["GrowthStageGroup"].value_counts(sort=True))

shape: (5, 2)
┌──────────────────┬────────┐
│ GrowthStageGroup ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ Early            ┆ 450374 │
│ null             ┆ 240900 │
│ Later            ┆ 112010 │
│ Exit             ┆ 69428  │
│ Out              ┆ 35222  │
└──────────────────┴────────┘


In [40]:
# ── 6.2 · lo stadio futuro, PRIMA del troncamento ─────────────────────────
# L'ordine conta: se si calcolasse dopo il troncamento, "Out" ed "Exit" non
# sarebbero piu' raggiungibili come stadio futuro - e il senso della colonna e'
# esattamente quello. E' l'errore piu' facile da fare riscrivendo questa fase.
panel = next_different(
    panel, "GrowthStageGroup", ["CompanyID"], "GrowthNextStageGroup", "TimeNextStageGroup"
)

# Quante righe restano all'azienda dopo questa: serve per le righe che non
# cambiano mai gruppo.
righe_rimanenti = pl.len().over("CompanyID") - pl.int_range(pl.len()).over("CompanyID") - 1
panel = panel.with_columns(
    # "Stay" NON e' un nullo: e' la stringa letterale che dice "questa azienda
    # non lascia il gruppo in cui e'". La fase 7 la sostituira' col gruppo corrente.
    pl.col("GrowthNextStageGroup").fill_null("Stay").alias("GrowthNextStageGroup"),
    # ...e la distanza e' quella dall'ULTIMA riga non troncata, non nulla.
    # Le due espressioni stanno nello stesso with_columns apposta: la seconda
    # legge GrowthNextStageGroup PRIMA che la prima lo riempia, perche' polars
    # valuta entrambe sul frame di partenza.
    pl.when(pl.col("GrowthNextStageGroup").is_null())
    .then(righe_rimanenti)
    .otherwise(pl.col("TimeNextStageGroup"))
    .alias("TimeNextStageGroup"),
)
print(panel["GrowthNextStageGroup"].value_counts(sort=True))

shape: (5, 2)
┌──────────────────────┬────────┐
│ GrowthNextStageGroup ┆ count  │
│ ---                  ┆ ---    │
│ str                  ┆ u32    │
╞══════════════════════╪════════╡
│ Stay                 ┆ 611703 │
│ Later                ┆ 119304 │
│ Out                  ┆ 111613 │
│ Exit                 ┆ 64969  │
│ Early                ┆ 345    │
└──────────────────────┴────────┘


In [41]:
# ── 6.3 · il troncamento all'uscita ───────────────────────────────────────
prima = panel.height
# cum_max su 0/1: vale 1 dalla prima riga terminale in poi. Si tengono solo le
# righe a 0, quindi anche la riga terminale stessa viene eliminata.
# NON e' una svista, ed e' cio' che rende corretto il target:
#   - il panel pubblicato non contiene nessuna riga con stadio Out o Exit;
#   - l'esito resta nel target attraverso GrowthNextStageGroup, che 6.2 calcola
#     PRIMA del troncamento: l'ultima riga rimasta dice "sta per uscire, fra N anni";
#   - tenendo la riga dell'uscita, TargetAge si sposta di +1 e cade proprio li',
#     dove lo stadio futuro punta alle righe DOPO l'uscita: verificato, 198
#     aziende verrebbero etichettate come non uscite (133 da Out a Early).
# Per questo l'ordine fra 6.2 e 6.3 e' un vincolo.
raggiunto_terminale = (
    pl.col("GrowthStageGroup")
    .is_in(GRUPPI_TERMINALI)
    .fill_null(False)
    .cast(pl.Int8)
    .cum_max()
    .over("CompanyID")
)
panel = panel.filter(raggiunto_terminale == 0)
# Elimina anche le righe NON terminali che seguono una terminale: 1.136 righe
# in 413 aziende (836 Early, 215 Later, 85 senza stadio). Sono aziende date
# per fallite che prendono un altro round, o acquisite che continuano a
# raccogliere: la loro vita dopo l'uscita scompare. E' coerente con il trattare
# l'uscita come uno stato assorbente, e va dichiarato.
# Scomposizione delle righe tolte: 39.698 l'anno dell'uscita, 64.952 anni
# successivi gia' terminali, 1.136 non terminali.
print(f"righe: {prima:,} -> {panel.height:,}  (attese 802.148)")
print(f"aziende: {panel['CompanyID'].n_unique():,}  (attese 116.312)")

righe: 907,934 -> 802,148  (attese 802.148)
aziende: 116,312  (attese 116.312)


---
## Fase 7 — competitor, anno per anno

Le tre colonne competitor — `N_Competitors`, `Same_Country`,
`SimilarityScoreMean` — esistono in **una sola versione**, il cui contenuto
dipende da `TEMPORIZZA_COMPETITOR`:

- **acceso**: si contano solo i concorrenti *vivi* in quell'anno, e la coppia
  entra solo se della controparte conosciamo la finestra di vita, che si legge
  soltanto da `Company.csv`. È la versione temporalmente pulita, e il prezzo è
  che sopravvive circa un decimo delle coppie dichiarate.
- **spento**: ogni informazione disponibile, senza finestra temporale e senza
  filtro sulla controparte, comprese le aziende simili fuori estrazione
  (incumbent, estere, fondate prima del 2000). È la baseline **con** il
  look-ahead.

A queste si affianca `N_Similar`, il numero di aziende simili attive su cui la
media di similarità è calcolata: serve a distinguere «nessun comparabile» da
«comparabili poco simili», che in `SimilarityScoreMean` sarebbero entrambi zero.

In [42]:
# ── 7.1 · la finestra di vita di OGNI azienda ─────────────────────────────
# Si rilegge Company.csv per intero, non solo le coorti del panel: un
# concorrente puo' essere piu' vecchio del 2000 e va comunque considerato vivo.
COLONNE_VITA = [*COMPANY_DATE_COLUMNS, "FiscalPeriod", "YearFounded", "HQCountry"]
tutte = as_na(read_raw(cfg, "Company", ["CompanyID", *COLONNE_VITA]), R_NA_NAN)

trimestre_v = pl.col("FiscalPeriod").str.extract(r"TTM (\d)Q\d{4}", 1).cast(pl.Int64, strict=False)
anno_v = pl.col("FiscalPeriod").str.slice(-4).cast(pl.Int64, strict=False)
tutte = tutte.with_columns(
    *[parse_date_r(pl.col(c)).alias(c) for c in COMPANY_DATE_COLUMNS],
    pl.date(anno_v, trimestre_v * 3, 30).alias("FiscalDate"),
    pl.col("YearFounded").cast(pl.Int64, strict=False),
)

# Stessa definizione di MaxYear della fase 1: l'ultimo anno con DATI.
# ASSUNZIONE, ed e' la piu' pesante di questa fase: MaxYear viene usato come
# "l'azienda era ancora viva". Non e' la data di chiusura, e un'azienda ben
# coperta da PitchBook risulta viva piu' a lungo di una coperta male: il
# conteggio dei concorrenti sovrappesa quindi i grandi e i ben documentati.
vita = (
    tutte.with_columns(
        pl.max_horizontal([pl.col(c).dt.year() for c in [*COMPANY_DATE_COLUMNS, "FiscalDate"]]).alias("MaxYear")
    )
    .select("CompanyID", "YearFounded", "MaxYear", "HQCountry")
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
)
del tutte
gc.collect()
print(f"aziende con una finestra di vita utilizzabile: {vita.height:,} su 134.355")

aziende con una finestra di vita utilizzabile: 126,817 su 134.355


In [43]:
# ── 7.2 · le coppie azienda / azienda simile ──────────────────────────────
simili_grezze = as_na(
    read_raw(cfg, "CompanySimilarRelation",
             ["CompanyID", "SimilarCompanyID", "SimilarityScore", "IsCompetitor",
              "SimilarCompanyHQCountry"]),
    R_NA_NAN,
).with_columns(to_num("SimilarityScore"))

# UNICO filtro sempre valido: l'azienda di sinistra deve stare nel panel.
id_panel = panel.select("CompanyID").unique().to_series()
simili_grezze = simili_grezze.filter(pl.col("CompanyID").is_in(id_panel.implode()))

# Il paese della controparte si legge dalla TABELLA DELLE RELAZIONI, non da
# Company.csv: c'e' nel 99,57% delle righe, e sulle 228.894 coppie in cui si
# possono confrontare le due fonti coincidono sul 100%. Cosi' Same_Country
# resta calcolabile anche quando la controparte e' fuori estrazione.
simili_grezze = (
    simili_grezze
    .join(vita.select("CompanyID", pl.col("HQCountry").alias("_paese_proprio")),
          on="CompanyID", how="left")
    .with_columns(
        # Nullo se manca uno dei due paesi: la coppia viene esclusa dal
        # conteggio, non contata come "paese diverso".
        (pl.col("_paese_proprio") == pl.col("SimilarCompanyHQCountry")).alias("_stesso_paese")
    )
    .drop("_paese_proprio", "SimilarCompanyHQCountry")
)

# Due insiemi diversi: "simili" (tutte le coppie, per la media di similarita')
# e "concorrenti" (solo IsCompetitor = "Yes", per i conteggi). La relazione e'
# ORIENTATA e reciproca solo nel 3,6% dei casi: ASSUNZIONE, si conta chi
# l'azienda dichiara, non chi dichiara lei.
simili = simili_grezze.select("CompanyID", "SimilarCompanyID", "SimilarityScore")
concorrenti = simili_grezze.filter(pl.col("IsCompetitor") == "Yes").select(
    "CompanyID", "SimilarCompanyID", "SimilarityScore", "_stesso_paese"
)

if TEMPORIZZA_COMPETITOR:
    # Sapere QUANDO la controparte era viva si legge solo da Company.csv, quindi
    # le coppie con la controparte fuori estrazione si perdono: e' il prezzo
    # della temporizzazione, e sopravvive circa un decimo delle coppie.
    finestra = vita.select(
        pl.col("CompanyID").alias("SimilarCompanyID"),
        pl.col("YearFounded").alias("YF"),
        pl.col("MaxYear").alias("MY"),
    )
    dichiarate = (concorrenti.height, simili.height)
    simili = simili.join(finestra, on="SimilarCompanyID", how="inner")
    concorrenti = concorrenti.join(finestra, on="SimilarCompanyID", how="inner")
    del finestra
    print(f"coppie dichiarate           : concorrenti {dichiarate[0]:,}   simili {dichiarate[1]:,}")
    print(f"coppie con finestra di vita : concorrenti {concorrenti.height:,}   simili {simili.height:,}")
else:
    print(f"coppie usate (nessun filtro): concorrenti {concorrenti.height:,}   simili {simili.height:,}")

del simili_grezze, vita
gc.collect()

coppie dichiarate           : concorrenti 116,081   simili 1,160,950
coppie con finestra di vita : concorrenti 27,729   simili 208,372


0

In [44]:
# ── 7.3 · chi era attivo in quale anno ────────────────────────────────────
anni_panel = panel.select("CompanyID", "Year_Delta").unique()

if TEMPORIZZA_COMPETITOR:
    # active_pairs e' un RANGE JOIN: per ogni (azienda, anno) del panel tiene le
    # coppie il cui concorrente era vivo quell'anno, cioe' YF <= anno <= MY.
    # join_where lo esegue come join di disuguaglianza invece di materializzare
    # il prodotto cartesiano: e' l'unico modo perche' ci stia in memoria, il
    # risultato e' gia' di diversi milioni di righe.
    #
    # MY e' MaxYear, l'ultimo evento registrato dell'azienda simile. Non e' la
    # sua data di morte, e il 52% delle aziende ancora operative ha MaxYear
    # prima del 2024: qualche concorrente vivo esce dalla finestra in anticipo.
    # Si tiene comunque, perche' MaxYear e' la fine della vita in TUTTA la
    # pipeline - lo scheletro stesso finisce li' - e cambiarlo solo qui
    # introdurrebbe un'incoerenza peggiore.
    attivi_concorrenti = active_pairs(anni_panel, concorrenti, ["_stesso_paese"])
    attivi_simili = active_pairs(anni_panel, simili, ["SimilarityScore"])
    print(f"coppie concorrente-anno attive: {attivi_concorrenti.height:,}")
    print(f"coppie simile-anno attive     : {attivi_simili.height:,}")
else:
    # Senza temporizzazione non c'e' niente da tagliare: ogni coppia vale per
    # ogni anno dell'azienda, e a replicarla ci pensa l'aggregazione di 7.4.
    attivi_concorrenti = attivi_simili = None
    print("TEMPORIZZA_COMPETITOR spento: nessun taglio per anno")

coppie concorrente-anno attive: 190,601
coppie simile-anno attive     : 904,351


In [45]:
# ── 7.4 · le tre colonne competitor ───────────────────────────────────────
# Qualunque sia il ramo il risultato ha la stessa forma, una riga per
# (azienda, anno): con la temporizzazione spenta il valore e' semplicemente lo
# stesso per tutti gli anni della stessa azienda.
if TEMPORIZZA_COMPETITOR:
    stat_concorrenti = attivi_concorrenti.group_by("CompanyID", "Year_Delta").agg(
        pl.col("SimilarCompanyID").n_unique().alias("N_Competitors"),
        # Same_Country e' un CONTEGGIO di concorrenti attivi nello stesso
        # paese dell'azienda, senza nessuna soglia di similarita'.
        pl.col("_stesso_paese").drop_nulls().sum().cast(pl.Int64).alias("Same_Country"),
    )
    stat_simili = attivi_simili.group_by("CompanyID", "Year_Delta").agg(
        pl.col("SimilarityScore").mean().alias("SimilarityScoreMean"),
        # Il DENOMINATORE di quella media: su quante aziende simili e' calcolata.
        # Senza, uno 0 in SimilarityScoreMean e' ambiguo - "nessun comparabile"
        # oppure "comparabili dissimili" - e nessun'altra colonna scioglie il
        # dubbio: N_Competitors conta un sottoinsieme, e dove vale 0 la media e'
        # vera nel 52% dei casi e fabbricata nel 48%.
        pl.col("SimilarCompanyID").n_unique().alias("N_Similar"),
    )
else:
    stat_concorrenti = anni_panel.join(
        concorrenti.group_by("CompanyID").agg(
            pl.col("SimilarCompanyID").n_unique().alias("N_Competitors"),
            pl.col("_stesso_paese").drop_nulls().sum().cast(pl.Int64).alias("Same_Country"),
        ),
        on="CompanyID", how="inner",
    )
    stat_simili = anni_panel.join(
        simili.group_by("CompanyID").agg(
            pl.col("SimilarityScore").mean().alias("SimilarityScoreMean"),
            pl.col("SimilarCompanyID").n_unique().alias("N_Similar"),
        ),
        on="CompanyID", how="inner",
    )

del attivi_concorrenti, attivi_simili, concorrenti, simili
gc.collect()
print(f"anni-azienda con almeno un concorrente: {stat_concorrenti.height:,}")

anni-azienda con almeno un concorrente: 98,409


In [46]:
# ── 7.5 · innestare, sostituire "Stay", rinumerare ────────────────────────
finale = (
    panel
    .join(stat_concorrenti, on=["CompanyID", "Year_Delta"], how="left")
    .join(stat_simili, on=["CompanyID", "Year_Delta"], how="left")
    .with_columns(
        # Nessun concorrente -> zero concorrenti. Corretto: e' un conteggio.
        pl.col("N_Competitors", "Same_Country", "N_Similar").fill_null(0),
        # Nessuna azienda simile -> similarita' media 0. Discutibile: zero e'
        # il MINIMO della scala, non un valore neutro, quindi un'azienda senza
        # simili appare a un modello come un'azienda i cui simili sono
        # massimamente diversi: per questo si aggiunge N_Similar.
        pl.col("SimilarityScoreMean").fill_null(0.0),
    )
    .with_columns(
        # "Stay" significa che l'azienda non lascia il gruppo in cui e': lo
        # stadio futuro e' quello corrente.
        pl.when(pl.col("GrowthNextStageGroup") == "Stay")
        .then(pl.col("GrowthStageGroup"))
        .otherwise(pl.col("GrowthNextStageGroup"))
        .alias("GrowthNextStageGroup")
    )
)
del panel, stat_concorrenti, stat_simili
gc.collect()

# La rinumerazione e' l'ULTIMA operazione della pipeline, e non per caso:
# distrugge ogni possibilita' di join con i file di riferimento.
mappa_id = finale.select("CompanyID").unique().sort("CompanyID").with_row_index("_nuovo", offset=1)
finale = finale.join(mappa_id, on="CompanyID", how="left").drop("CompanyID").rename({"_nuovo": "CompanyID"})

# Le colonne di example_panel, nello stesso ordine, con TotalRaised al posto di
# TotalRaised_Est e SENZA le tre _All: le colonne competitor sono una sola
# terna, ed e' TEMPORIZZA_COMPETITOR a deciderne il contenuto. In coda
# UndisclosedAmountShare. Year_Delta ha fatto da chiave fino a qui e si ferma.
COLONNE_FINALI = [
    "CompanyID", "Age", "YearFounded",
    "GrowthStageGroup", "GrowthNextStageGroup", "TimeNextStageGroup",
    "N_Deal", "TotalRaised", "Percent_Females",
    "Is_Eco", "Is_Eng", "Is_NS", "Is_Hum", "Is_SS", "Is_Med", "Is_Law", "Is_IT",
    "Institute", "WorkExp_Idx_Mean", "Total_Founders",
    "Is_Debt", "Is_SpinOff", "Is_CrowdFunding", "MeanMedianRoundAmount_cum",
    "Is_Accelerator", "has_Corporate", "has_VentureCapital", "has_PublicInvestor",
    "has_Angel_Lead", "has_Corporate_Lead", "has_VentureCapital_Lead",
    "has_Accelerator_Lead", "has_PrivateEquity_Lead", "has_PublicInvestor_Lead",
    "HQCountry", "PrimaryIndustrySector",
    "SimilarityScoreMean", "N_Competitors", "Same_Country",
    "Highest_Degree_CEO", "Gender_CEO", "MeanTotalInvestments_cum", "WorkExperienceIndex_CEO",
    "Is_Angel", "Total_People", "Is_Grant", "has_PrivateEquity", "TotalInvestors",
    "Highest_Degree_Mean", "Avg_Earliest_Year",
    # Le due colonne che non vengono da example_panel. Tutte e due sono
    # ADDITIVE: eliminarle riporta esattamente allo schema precedente.
    "UndisclosedAmountShare",   # l'indicatore aggiunto in 4.8
    "N_Similar",                # il denominatore di SimilarityScoreMean, vedi 7.4
]
finale = finale.select(COLONNE_FINALI)

finale.write_parquet(cfg.interim("panel.parquet"))
finale.write_csv(cfg.interim("panel.csv.gz"), compression="gzip")
print(f"panel finale: {finale.height:,} righe x {finale.width} colonne")
print(f"'Stay' residui: {(finale['GrowthNextStageGroup'] == 'Stay').sum()}   (deve essere 0)")

panel finale: 802,148 righe x 52 colonne
'Stay' residui: 0   (deve essere 0)


---
## Verifica

L'ultima cella controlla le invarianti del panel: numero di righe, di aziende e
di colonne, quante righe hanno dati di team e uno stadio di crescita, più due
controlli di merito (nessun anno precedente alla fondazione, e la coorte più
vecchia deve avere dati di team).

I valori attesi sono legati a **questa estrazione**: uno scostamento significa
«è cambiato qualcosa, capisci cosa prima di usare il risultato», non
necessariamente «c'è un errore».

In [47]:
# ── Verifica delle invarianti ─────────────────────────────────────────────
finale = pl.read_parquet(cfg.interim("panel.parquet"))

ATTESI = {
    "righe": 802_148,
    "aziende": 116_312,
    "colonne": 52,
    "righe con dati di team": 749_847,
    "righe con stadio": 561_333,
}
ottenuti = {
    "righe": finale.height,
    "aziende": finale["CompanyID"].n_unique(),
    "colonne": finale.width,
    "righe con dati di team": int(finale["Total_People"].is_not_null().sum()),
    "righe con stadio": int(finale["GrowthStageGroup"].is_not_null().sum()),
}
for nome, atteso in ATTESI.items():
    ott = ottenuti[nome]
    stato = "ok" if ott == atteso else f"ATTESO {atteso:,}"
    print(f"  {nome:<24}{ott:>10,}   {stato}")

# Due invarianti di merito, oltre ai conteggi.
assert finale.filter(pl.col("Age") < 0).height == 0, "ci sono anni precedenti alla fondazione"
coorte2000 = finale.filter(pl.col("YearFounded") == 2000)
quota = 100 * coorte2000["Total_People"].is_not_null().sum() / coorte2000.height
assert quota > 80, f"la coorte 2000 ha solo il {quota:.1f}% di righe con team"
print(f"\n  righe con Age < 0            0   ok")
print(f"  coorte 2000 con team      {quota:5.1f}%  ok")

# Le colonne devono essere esattamente quelle di example_panel, con
# TotalRaised al posto di TotalRaised_Est.
esempio = pl.read_csv("data/raw/example_panel.csv", infer_schema_length=0).columns
# Le tre colonne _All non esistono piu': la terna competitor e' una sola,
# e TEMPORIZZA_COMPETITOR ne decide il contenuto.
atteso_colonne = [c if c != "TotalRaised_Est" else "TotalRaised"
                  for c in esempio if not c.endswith("_All")]
atteso_colonne += ["UndisclosedAmountShare", "N_Similar"]   # aggiunte in 4.8 e 7.4
assert finale.columns == atteso_colonne, "le colonne non coincidono con example_panel"
print("  colonne: le 50 di example_panel senza le _All, piu' le 2 aggiunte   ok")

if all(ottenuti[k] == v for k, v in ATTESI.items()):
    print("\nTUTTE LE INVARIANTI RISPETTATE.")
else:
    print("\nQUALCOSA E' CAMBIATO: confronta con docs/panel_revisione_stato.md.")

  righe                      802,148   ok
  aziende                    116,312   ok
  colonne                         52   ok
  righe con dati di team     749,847   ok
  righe con stadio           561,333   ok

  righe con Age < 0            0   ok
  coorte 2000 con team       92.3%  ok


  colonne: le 50 di example_panel senza le _All, piu' le 2 aggiunte   ok

TUTTE LE INVARIANTI RISPETTATE.
